# Whole-brain causal connectivity during decoded neurofeedback: main analysis

Reproduces the computed panels of the main-text figures and Supplementary Figures 6-10 of
Arab, Ghassami, Jamalabadi, Peters & Nozari, *Whole-brain causal connectivity during decoded neurofeedback: a meta-study*.

| Figure | Content |
|---|---|
| Fig. 1c-e | similarity hierarchy, NF vs baseline similarity, graph asymmetry |
| Fig. 2a-i | NF vs baseline edges, degree, net flow, global measures |
| Fig. 3b-c | edges modulated in NF and their correlation with score; control-hub parcels |
| Fig. 4a-d | cognitive vs perceptive neurofeedback |
| Supp. Figs. 6-10 | average partial-correlation graphs, control hub with DMN, score correlations, score distributions, cognitive vs perceptive global and nodal measures |

Figs. 1a, 1b and 3a are schematics. Supplementary Fig. 21 is made by `supp_modified_calltif_asymmetry.ipynb`; the other supplementary figures by the scripts in `4_robustness/`.

**Inputs** (see the README): the CaLLTiF graphs from `2_causal_graphs/`, the parcellated time series from `1_preprocessing/`, and the neurofeedback scores (`in_score_d*.csv`) of the DecNef Collection in `Results/glm_revision/scores/Study{N}/sub-XX/pattern/`.
**Outputs:** `Results/figures_main/` and `Results/figures_supplementary/`.


In [ ]:
# Run from the repository root so that 'Results/...' paths and the shared modules resolve
import os, sys
ROOT = os.path.dirname(os.path.abspath(''))
os.chdir(ROOT)
sys.path.insert(0, ROOT)

# Output folders for the figures
FIG_MAIN = 'Results/figures_main'
FIG_SUPP = 'Results/figures_supplementary'
os.makedirs(FIG_MAIN, exist_ok=True)
os.makedirs(FIG_SUPP, exist_ok=True)

# Import Packages

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import csv
import pandas as pd
from scipy.stats import pearsonr
from scipy.spatial import distance_matrix
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from scipy import stats
import seaborn as sb
import glob
from scipy.io import savemat
import importlib
import itertools
import scipy.io
import networkx as nx
import os
import datetime
import time
import sys
import warnings
import seaborn as sns
import functions as funcs
from sklearn.metrics import pairwise_distances
import matplotlib.colors as mcolors
from matplotlib.ticker import PercentFormatter
# import netplotbrain
from nilearn.connectome import ConnectivityMeasure
from nilearn import plotting
import matplotlib.cm as cm
import matplotlib.colors as colors
import nilearn
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
import itertools
from matplotlib.colors import BoundaryNorm
import pingouin as pg
from statsmodels.stats.multitest import multipletests
import statsmodels
import math
from matplotlib.patches import FancyArrowPatch, Circle
from matplotlib.patches import Rectangle
from nilearn import plotting
import webcolors

importlib.reload(funcs)
sys.path.insert(0, 'External Packages and Files/tigramite-master')

# Variables

In [ ]:
num_parcels = 116
num_cortical_parcels = 100
num_subcortical_parcels = 16

num_nf_sessions = 3
num_decoder_sessions = 1
subject_id_selected = np.array(['01','02','03','04','05','06','07','08','09']) 
num_subjects = subject_id_selected.shape[0]

parcellation_mask = np.asanyarray(nib.load('External Packages and Files/Schaefer2018_100Parcels_7Networks_order_Tian_Subcortex_S1.dlabel.nii').dataobj)

# network
num_networks = 16

#subcortical
# visual
# limbic
# control
# default
# ventral
# dorsal
# somatomotor

network_dic = "{'LH_Sub':[],'RH_Sub':[],'LH_Vis' :[],'RH_Vis' :[], 'LH_Lim':[],'RH_Lim':[],'LH_Con':[],'RH_Con':[],'LH_Def':[],'RH_Def':[],'LH_Sal':[],'RH_Sal':[],'LH_Dor':[],'RH_Dor':[],'LH_Som':[],'RH_Som':[]}"
net_labels = ['LH_Sub','RH_Sub','LH_Vis','RH_Vis', 'LH_Lim','RH_Lim','LH_Con','RH_Con','LH_Def','RH_Def','LH_Sal','RH_Sal','LH_Dor','RH_Dor','LH_Som','RH_Som']
visualization_labels = ['L-Sub','R-Sub','L-Vis','R-Vis','L-Lim','R-Lim','L-Cont','R-Cont','L-DMN','R-DMN','L-Vent-Att','R-Vent-Att','L-Dors-Att','R-Dors-Att','L-Som-Mot','R-Som-Mot']
short_label = ['Sub','Sub','Vis','Vis','Lim','Lim','Cont','Cont','DMN','DMN','Vent-Att','Vent-Att','Dors-Att','Dors-Att','Som-Mot','Som-Mot']

short_label_modified = short_label
for i,s in enumerate(short_label_modified):
    if s == 'Dors-Att':
        short_label_modified[i] = 'Dors\n Att'
    if s== 'Vent-Att':
        short_label_modified[i] = 'Vent\n Att'
    if s == 'Som-Mot':
        short_label_modified[i] = 'Som\n Mot'

print(short_label_modified)

num_networks = len(net_labels)

with open('Schaefer2018_100Parcels_7Networks_order_Tian_Subcortex_S1_label.txt') as f:
    lines = f.readlines()

parcel_labels = []
parcel_locs = []
i = 0
while i< 2*num_parcels:
    if i> 31:
        parcel_labels.append(lines[i].strip('\n')[10:])
    else:
        parcel_labels.append(lines[i].strip('\n'))
    i = i+2

print(parcel_labels)

# CaLLTiF
alpha_level = 1e-2/4 # tau_max = 1 ----> alpha/(tau_max+1)*2^(tau_max)
tau_max = 1

num_subjects_ready = 9

num_samples_in_each_run = 154      #175
num_trials_each_run = 15

num_dec_runs  = 5      ### five consecutive pairs of runs (6 runs)
num_nf_runs = 7        ### seven consecutive pairs of runs (8 runs)

num_studies = 5

# Functions

In [ ]:
def import_npz(npz_file):
    Data = np.load(npz_file, allow_pickle= True)
    for varName in Data:
        print(varName)
        globals()[varName] = Data[varName]  

In [ ]:
num_networks = len(net_labels)
num_parcels = len(parcel_labels)

network_parcel_idx = eval(network_dic)
network_parcel_labels = eval(network_dic)

for i in range (0 , num_parcels):

    if i < 8:
        network_parcel_idx['RH_Sub'].append(i)
        network_parcel_labels['RH_Sub'].append(parcel_labels[i])

    elif i>=8 and i<16:
        network_parcel_idx['LH_Sub'].append(i)
        network_parcel_labels['LH_Sub'].append(parcel_labels[i])

    elif i>=16:
        temp = parcel_labels[i]
        temp_first = temp[0:6]
        if temp_first in network_parcel_idx:
            network_parcel_idx[temp_first].append(i)
            network_parcel_labels[temp_first].append(parcel_labels[i])

print(network_parcel_idx)
print(network_parcel_labels)

print(network_parcel_labels['LH_Con'])
print(network_parcel_labels['RH_Con'])

In [ ]:
parcel_labels_rearranged = []

for n in network_parcel_labels:

    parcel_labels_rearranged = parcel_labels_rearranged + network_parcel_labels[n]

parcel_labels_rearranged

# Load CaLLTiF Results

In [ ]:
import_npz('Results/calltif_decnef_study1_dec_two_runs_one_graph.npz')
import_npz('Results/calltif_decnef_study1_nf_two_runs_one_graph.npz')

import_npz('Results/calltif_decnef_study2_dec_two_runs_one_graph.npz')
import_npz('Results/calltif_decnef_study2_nf_two_runs_one_graph.npz')

import_npz('Results/calltif_decnef_study3_dec_two_runs_one_graph.npz')
import_npz('Results/calltif_decnef_study3_nf_two_runs_one_graph.npz')

import_npz('Results/calltif_decnef_study4_dec_two_runs_one_graph.npz')
import_npz('Results/calltif_decnef_study4_nf_two_runs_one_graph.npz')

import_npz('Results/calltif_decnef_study5_dec_two_runs_one_graph.npz')
import_npz('Results/calltif_decnef_study5_nf_two_runs_one_graph.npz')

#study1
import_npz('Results/study1_preprocessed_confounders_removed_selected_subjects.npz')
num_runs_in_each_session_study1_nf = num_runs_all

import_npz('Results/study1_preprocessed_confounders_removed_selected_subjects_decoder_session0.npz')
num_runs_in_each_session_study1_dec = num_runs_all

#study2
import_npz('Results/study2_preprocessed_confounders_removed_selected_subjects.npz')
num_runs_in_each_session_study2_nf = num_runs_all

import_npz('Results/study2_preprocessed_confounders_removed_selected_subjects_decoder_session0.npz')
num_runs_in_each_session_study2_dec = num_runs_all

#study3
import_npz('Results/study3_preprocessed_confounders_removed_selected_subjects.npz')
num_runs_in_each_session_study3_nf = num_runs_all

import_npz('Results/study3_preprocessed_confounders_removed_selected_subjects_decoder_session0.npz')
num_runs_in_each_session_study3_dec = num_runs_all

#study4
import_npz('Results/study4_preprocessed_confounders_removed_selected_subjects.npz')
num_runs_in_each_session_study4_nf = num_runs_all

import_npz('Results/study4_preprocessed_confounders_removed_selected_subjects_decoder_session0.npz')
num_runs_in_each_session_study4_dec = num_runs_all

#study5
import_npz('Results/study5_preprocessed_confounders_removed_selected_subjects.npz')
num_runs_in_each_session_study5_nf = num_runs_all

import_npz('Results/study5_preprocessed_confounders_removed_selected_subjects_decoder_session0.npz')
num_runs_in_each_session_study5_dec = num_runs_all

# Load Scores

In [ ]:
study1_subject_id_selected = np.array(['03','24','21','15','23','05','17','13','18']) 
study1_num_subjects= study1_subject_id_selected.shape[0]

study1_scores = [[[0 for ses in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,study1_num_subjects)]

for sub in range(0, study1_num_subjects):
    path_to_pattern =  'Results/glm_revision/scores/Study1/sub-%s/pattern'%study1_subject_id_selected[sub]
    print('subject:', study1_subject_id_selected[sub])
    for ses in range(0,num_nf_sessions):
        df = pd.read_csv(os.path.join(path_to_pattern,'in_score_d%d.csv'%(ses+1)))
        scores_for_8_runs = df.values[0:120]

        for r in range(0, num_nf_runs):

            study1_scores[sub][ses][r] = np.concatenate((scores_for_8_runs[r*num_trials_each_run:(r+1)*num_trials_each_run], scores_for_8_runs[(r+1)*num_trials_each_run:(r+2)*num_trials_each_run]))

In [ ]:
study2_subject_id_selected = np.array(['01','02','03','04','05','06','07','08','09']) 
study2_num_subjects= study2_subject_id_selected.shape[0]

study2_scores = [[[0 for ses in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,study2_num_subjects)]

for sub in range(0, study2_num_subjects):
    path_to_pattern =  'Results/glm_revision/scores/Study2/sub-%s/pattern'%study2_subject_id_selected[sub]

    for ses in range(0,num_nf_sessions):
        df = pd.read_csv(os.path.join(path_to_pattern,'in_score_d%d.csv'%(ses+1)))
        scores_for_8_runs = df.values[0:120]

        for r in range(0, num_nf_runs):

            study2_scores[sub][ses][r] = np.concatenate((scores_for_8_runs[r*num_trials_each_run:(r+1)*num_trials_each_run], scores_for_8_runs[(r+1)*num_trials_each_run:(r+2)*num_trials_each_run]))

In [ ]:
study3_subject_id_selected = np.array(['01','05','08','10','12','13','14','15','16'])
study3_num_subjects= study3_subject_id_selected.shape[0]

study3_scores = [[[0 for ses in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,study3_num_subjects)]
for sub in range(0, study3_num_subjects):
    path_to_pattern =  'Results/glm_revision/scores/Study3/sub-%s/pattern'%study3_subject_id_selected[sub]

    for ses in range(0,num_nf_sessions):
        df = pd.read_csv(os.path.join(path_to_pattern,'in_score_d%d.csv'%(ses+1)))
        scores_for_8_runs = df.values[0:120]

        for r in range(0, num_nf_runs):

            study3_scores[sub][ses][r] = np.concatenate((scores_for_8_runs[r*num_trials_each_run:(r+1)*num_trials_each_run], scores_for_8_runs[(r+1)*num_trials_each_run:(r+2)*num_trials_each_run]))

In [ ]:
study4_subject_id_selected = np.array(['01','02','03','04','05','06','07','08','09']) 
study4_num_subjects= study4_subject_id_selected.shape[0]

study4_scores = [[[0 for ses in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,study4_num_subjects)]

for sub in range(0, study4_num_subjects):
    path_to_pattern =  'Results/glm_revision/scores/Study4/sub-%s/pattern'%study4_subject_id_selected[sub]
    for ses in range(0,num_nf_sessions):
        df = pd.read_csv(os.path.join(path_to_pattern,'in_score_d%d.csv'%(ses+2)))
        scores_for_8_runs = df.values[0:120]

        for r in range(0, num_nf_runs):

            study4_scores[sub][ses][r] = np.concatenate((scores_for_8_runs[r*num_trials_each_run:(r+1)*num_trials_each_run], scores_for_8_runs[(r+1)*num_trials_each_run:(r+2)*num_trials_each_run]))

In [ ]:
study5_subject_id_selected = np.array(['01','02','03','06','08','09','11','12','15']) 
study5_num_subjects= study5_subject_id_selected.shape[0]

study5_scores = [[[0 for ses in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,study5_num_subjects)]
for sub in range(0, study5_num_subjects):
    path_to_pattern =  'Results/glm_revision/scores/Study5/sub-%s/pattern'%study5_subject_id_selected[sub]

    for ses in range(0,num_nf_sessions):
        df = pd.read_csv(os.path.join(path_to_pattern,'in_score_d%d.csv'%(ses+2)))
        scores_for_8_runs = df.values[0:120]
        for r in range(0, num_nf_runs):
            study5_scores[sub][ses][r] = np.concatenate((scores_for_8_runs[r*num_trials_each_run:(r+1)*num_trials_each_run], scores_for_8_runs[(r+1)*num_trials_each_run:(r+2)*num_trials_each_run]))

## Distribution of scores in each study (Supplementary Fig. 9)

In [ ]:
# ... (Your data loading code remains the same) ...

studies = [study1_scores, study2_scores, study3_scores, study4_scores, study5_scores]
labels = ['Study 1', 'Study 2', 'Study 3', 'Study 4', 'Study 5']

data_frames = []
for data, label in zip(studies, labels):
    flat_data = np.array(data).flatten()
    df_temp = pd.DataFrame({'Scores': flat_data, 'Study': label})
    data_frames.append(df_temp)

df_final = pd.concat(data_frames, ignore_index=True)

plt.figure(figsize=(14, 9))
nice_colors = ["#FF6F61", "#6B5B95", "#88B04B", "#F7CAC9", "#92A8D1"]

# 1. The Violin Plot (Background)
# inner=None removes the default box so we can draw a custom one
ax = sns.violinplot(
    data=df_final, 
    x='Study', 
    y='Scores', 
    hue='Study', 
    palette=nice_colors, 
    linewidth=2.5, 
    legend=False,
    cut=0,
    inner=None
)

# 2. The Box Plot (Middle)
# Draws a clean white box to show median & quartiles
# sns.boxplot(
#     data=df_final, 
#     x='Study', 
#     y='Scores', 
#     width=0.1, 
#     color='white', 
#     ax=ax, 
#     boxprops={'zorder': 2, 'edgecolor': 'black', 'linewidth': 1.5},
#     medianprops={'color': 'black', 'linewidth': 2},
#     fliersize=0
# )

# 3. The Strip Plot (Points)
# Adds the individual dots with transparency
sns.stripplot(
    data=df_final, 
    x='Study', 
    y='Scores', 
    color='black', 
    size=2, 
    jitter=0.2, 
    alpha=0.3, 
    ax=ax
)

plt.xlabel('')
plt.xticks(fontsize=30)
# Added yticks size to match the "large font" requirement
plt.yticks(fontsize=30) 
plt.ylabel('Neurofeedback Score Value', fontsize=32)

sns.despine()

plt.savefig(os.path.join(FIG_SUPP, 'SuppFig09_score_distributions.pdf'), bbox_inches='tight')

plt.show()

# Score preprocessing: logit, z-score within study, average per graph (Fig. 3a)

In [ ]:
from scipy.special import logit, expit

study1_scores_inv_sigmoid_all = [[[0 for r in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,num_subjects_ready)]
study2_scores_inv_sigmoid_all = [[[0 for r in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,num_subjects_ready)]
study3_scores_inv_sigmoid_all = [[[0 for r in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,num_subjects_ready)]
study4_scores_inv_sigmoid_all = [[[0 for r in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,num_subjects_ready)]
study5_scores_inv_sigmoid_all = [[[0 for r in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,num_subjects_ready)]

### compute inverse of sigmoid

for sub in range(0, num_subjects_ready):
    for ses in range(num_nf_sessions):
        for r in range(0,num_nf_runs):

            study1_scores_inv_sigmoid = logit(np.array(study1_scores)[sub,ses,r,:,:])
            study1_scores_inv_sigmoid[np.isinf(study1_scores_inv_sigmoid)] = np.nan
            study1_scores_inv_sigmoid_all[sub][ses][r] = study1_scores_inv_sigmoid

            study2_scores_inv_sigmoid = logit(np.array(study2_scores)[sub,ses,r,:,:])
            study2_scores_inv_sigmoid[np.isinf(study2_scores_inv_sigmoid)] = np.nan
            study2_scores_inv_sigmoid_all[sub][ses][r] = study2_scores_inv_sigmoid
            
            study3_scores_inv_sigmoid = logit(np.array(study3_scores)[sub,ses,r,:,:])
            study3_scores_inv_sigmoid[np.isinf(study3_scores_inv_sigmoid)] = np.nan
            study3_scores_inv_sigmoid_all[sub][ses][r] = study3_scores_inv_sigmoid

            study4_scores_inv_sigmoid = logit(np.array(study4_scores)[sub,ses,r,:,:])
            study4_scores_inv_sigmoid[np.isinf(study4_scores_inv_sigmoid)] = np.nan
            study4_scores_inv_sigmoid_all[sub][ses][r] = study4_scores_inv_sigmoid
            
            study5_scores_inv_sigmoid = logit(np.array(study5_scores)[sub,ses,r,:,:])
            study5_scores_inv_sigmoid[np.isinf(study5_scores_inv_sigmoid)] = np.nan
            study5_scores_inv_sigmoid_all[sub][ses][r] = study5_scores_inv_sigmoid

#### zscore 
study1_score_mean = np.nanmean(np.array(study1_scores_inv_sigmoid_all))
study1_score_std = np.nanstd(np.array(study1_scores_inv_sigmoid_all))
study1_scores_inv_sigmoid_zscored = (np.array(study1_scores_inv_sigmoid_all)-study1_score_mean)/study1_score_std

study2_score_mean = np.nanmean(np.array(study2_scores_inv_sigmoid_all))
study2_score_std = np.nanstd(np.array(study2_scores_inv_sigmoid_all))
study2_scores_inv_sigmoid_zscored = (np.array(study2_scores_inv_sigmoid_all)-study2_score_mean)/study2_score_std

study3_score_mean = np.nanmean(np.array(study3_scores_inv_sigmoid_all))
study3_score_std = np.nanstd(np.array(study3_scores_inv_sigmoid_all))
study3_scores_inv_sigmoid_zscored = (np.array(study3_scores_inv_sigmoid_all)-study3_score_mean)/study3_score_std

study4_score_mean = np.nanmean(np.array(study4_scores_inv_sigmoid_all))
study4_score_std = np.nanstd(np.array(study4_scores_inv_sigmoid_all))
study4_scores_inv_sigmoid_zscored = (np.array(study4_scores_inv_sigmoid_all)-study4_score_mean)/study4_score_std

study5_score_mean = np.nanmean(np.array(study5_scores_inv_sigmoid_all))
study5_score_std = np.nanstd(np.array(study5_scores_inv_sigmoid_all))
study5_scores_inv_sigmoid_zscored = (np.array(study5_scores_inv_sigmoid_all)-study5_score_mean)/study5_score_std

#### Average score across run

study1_scores_avg = np.nanmean(study1_scores_inv_sigmoid_zscored, axis = 3)
study2_scores_avg = np.nanmean(study2_scores_inv_sigmoid_zscored, axis = 3)
study3_scores_avg = np.nanmean(study3_scores_inv_sigmoid_zscored, axis = 3)
study4_scores_avg = np.nanmean(study4_scores_inv_sigmoid_zscored, axis = 3)
study5_scores_avg = np.nanmean(study5_scores_inv_sigmoid_zscored, axis = 3)

study1_scores_change = [[[0 for r in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,num_subjects_ready)]
study2_scores_change = [[[0 for r in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,num_subjects_ready)]
study3_scores_change = [[[0 for r in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,num_subjects_ready)]
study4_scores_change = [[[0 for r in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,num_subjects_ready)]
study5_scores_change = [[[0 for r in range(0,num_nf_runs)]  for ses in range(0,num_nf_sessions)] for sub in range(0,num_subjects_ready)]

for sub in range(0, num_subjects_ready):
    for ses in range(num_nf_sessions):
        for r in range(0,num_nf_runs):
            study1_scores_change[sub][ses][r] = funcs.compute_average_score_change(study1_scores_inv_sigmoid_zscored[sub, ses,r,:,0])
            study2_scores_change[sub][ses][r]= funcs.compute_average_score_change(study2_scores_inv_sigmoid_zscored[sub, ses,r,:,0])
            study3_scores_change[sub][ses][r] = funcs.compute_average_score_change(study3_scores_inv_sigmoid_zscored[sub, ses,r,:,0])
            study4_scores_change[sub][ses][r] = funcs.compute_average_score_change(study4_scores_inv_sigmoid_zscored[sub, ses,r,:,0])
            study5_scores_change[sub][ses][r]= funcs.compute_average_score_change(study5_scores_inv_sigmoid_zscored[sub, ses,r,:,0])

# CaLLTiF Graphs

## Baseline (decoder-construction) sessions

In [ ]:
study1_decoder_time_series_graph =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study1_decoder_time_series_graph_pvalues =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study1_decoder_time_series_graph_statistics =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study1_decoder_binary_graph =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study_1_decoder_p_values_min = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study_1_decoder_statistics_max = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study1_decoder_subnetwork_graphs = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study1_decoder_partial_corr_subnetwork_graphs = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 

study2_decoder_time_series_graph =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study2_decoder_time_series_graph_pvalues =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study2_decoder_time_series_graph_statistics =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study2_decoder_binary_graph =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study_2_decoder_p_values_min = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study_2_decoder_statistics_max = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study2_decoder_subnetwork_graphs = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study2_decoder_partial_corr_subnetwork_graphs = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 

study3_decoder_time_series_graph =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study3_decoder_time_series_graph_pvalues =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study3_decoder_time_series_graph_statistics =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study3_decoder_binary_graph = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study_3_decoder_p_values_min = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study_3_decoder_statistics_max = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study3_decoder_subnetwork_graphs = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study3_decoder_partial_corr_subnetwork_graphs = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 

study4_decoder_time_series_graph =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study4_decoder_time_series_graph_pvalues =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study4_decoder_time_series_graph_statistics =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study4_decoder_binary_graph =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study_4_decoder_p_values_min = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study_4_decoder_statistics_max = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study4_decoder_subnetwork_graphs = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study4_decoder_partial_corr_subnetwork_graphs = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 

study5_decoder_time_series_graph =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study5_decoder_time_series_graph_pvalues =  [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study5_decoder_time_series_graph_statistics = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study5_decoder_binary_graph = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study_5_decoder_p_values_min = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study_5_decoder_statistics_max = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study5_decoder_subnetwork_graphs = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 
study5_decoder_partial_corr_subnetwork_graphs = [[ [ 0 for i in range(num_dec_runs)] for i in range(num_decoder_sessions)] for j in range(num_subjects_ready)] 

## decoder sessions
for sub in range(0, num_subjects_ready):
    for ses in range(0, num_decoder_sessions):
        for r in range(0, num_dec_runs):
            
            # study 1
            graph_dec_1 = multipledataset_calltif_decoder_study1_link_matrix[sub][ses][r]['graph']
            p_matrix_dec_1 = np.array(multipledataset_calltif_decoder_study1_link_matrix[sub][ses][r]['p_matrix'])
            statistics_dec_1 = np.array(multipledataset_calltif_decoder_study1_link_matrix[sub][ses][r]['val_matrix'])

            final_graph_new = p_matrix_dec_1 <= alpha_level
            graph = funcs.convert_to_string_graph(final_graph_new)
            study1_decoder_time_series_graph[sub][ses][r] = graph
            study1_decoder_time_series_graph_pvalues[sub][ses][r] = p_matrix_dec_1
            study1_decoder_time_series_graph_statistics[sub][ses][r] = statistics_dec_1

            binary_matrix , binary_matrix_p_value, weighted_matrix_statistics = funcs.summarize_across_all_lags_v2_with_pvalues_more_directed_v2(graph, p_matrix_dec_1,statistics_dec_1, alpha_level)
            study_1_decoder_p_values_min[sub][ses][r] = binary_matrix_p_value
            study_1_decoder_statistics_max[sub][ses][r] = funcs.rearange_matrix_v2(weighted_matrix_statistics, network_parcel_idx)
            study1_decoder_binary_graph[sub][ses][r] = funcs.rearange_matrix_v2(binary_matrix, network_parcel_idx)
            study1_decoder_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(binary_matrix,net_labels, network_dic, parcel_labels)
            study1_decoder_partial_corr_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(weighted_matrix_statistics,net_labels, network_dic, parcel_labels)

            # study 2
            graph_dec_2 = multipledataset_calltif_decoder_study2_link_matrix[sub][ses][r]['graph']
            p_matrix_dec_2 = np.array(multipledataset_calltif_decoder_study2_link_matrix[sub][ses][r]['p_matrix'])
            statistics_dec_2 = np.array(multipledataset_calltif_decoder_study2_link_matrix[sub][ses][r]['val_matrix'])

            final_graph_new = p_matrix_dec_2 <= alpha_level
            graph = funcs.convert_to_string_graph(final_graph_new)
            study2_decoder_time_series_graph[sub][ses][r] = graph
            study2_decoder_time_series_graph_pvalues[sub][ses][r] = p_matrix_dec_2
            study2_decoder_time_series_graph_statistics[sub][ses][r] = statistics_dec_2

            binary_matrix , binary_matrix_p_value, weighted_matrix_statistics = funcs.summarize_across_all_lags_v2_with_pvalues_more_directed_v2(graph, p_matrix_dec_2,statistics_dec_2, alpha_level)
            study_2_decoder_p_values_min[sub][ses][r] = binary_matrix_p_value
            study_2_decoder_statistics_max[sub][ses][r]=funcs.rearange_matrix_v2(weighted_matrix_statistics, network_parcel_idx)
            study2_decoder_binary_graph[sub][ses][r] = funcs.rearange_matrix_v2(binary_matrix, network_parcel_idx)
            study2_decoder_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(binary_matrix,net_labels, network_dic, parcel_labels)
            study2_decoder_partial_corr_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(weighted_matrix_statistics,net_labels, network_dic, parcel_labels)

            # study 3
            graph_dec_3 = multipledataset_calltif_decoder_study3_link_matrix[sub][ses][r]['graph']
            p_matrix_dec_3 = np.array(multipledataset_calltif_decoder_study3_link_matrix[sub][ses][r]['p_matrix'])
            statistics_dec_3 = np.array(multipledataset_calltif_decoder_study3_link_matrix[sub][ses][r]['val_matrix'])

            final_graph_new = p_matrix_dec_3 <= alpha_level
            graph = funcs.convert_to_string_graph(final_graph_new)
            study3_decoder_time_series_graph[sub][ses][r] = graph
            study3_decoder_time_series_graph_pvalues[sub][ses][r] = p_matrix_dec_3
            study3_decoder_time_series_graph_statistics[sub][ses][r] = statistics_dec_3

            binary_matrix , binary_matrix_p_value, weighted_matrix_statistics = funcs.summarize_across_all_lags_v2_with_pvalues_more_directed_v2(graph, p_matrix_dec_3,statistics_dec_3, alpha_level)
            study_3_decoder_p_values_min[sub][ses][r] = binary_matrix_p_value
            study_3_decoder_statistics_max[sub][ses][r] = funcs.rearange_matrix_v2(weighted_matrix_statistics, network_parcel_idx)
            study3_decoder_binary_graph[sub][ses][r] = funcs.rearange_matrix_v2(binary_matrix, network_parcel_idx)
            study3_decoder_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(binary_matrix,net_labels, network_dic, parcel_labels)
            study3_decoder_partial_corr_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(weighted_matrix_statistics,net_labels, network_dic, parcel_labels)

            # study 4
            graph_dec_4 = multipledataset_calltif_decoder_study4_link_matrix[sub][ses][r]['graph']
            p_matrix_dec_4 = np.array(multipledataset_calltif_decoder_study4_link_matrix[sub][ses][r]['p_matrix'])
            statistics_dec_4 = np.array(multipledataset_calltif_decoder_study4_link_matrix[sub][ses][r]['val_matrix'])

            final_graph_new = p_matrix_dec_4 <= alpha_level
            graph = funcs.convert_to_string_graph(final_graph_new)
            study4_decoder_time_series_graph[sub][ses][r] = graph
            study4_decoder_time_series_graph_pvalues[sub][ses][r] = p_matrix_dec_4
            study4_decoder_time_series_graph_statistics[sub][ses][r] = statistics_dec_4

            binary_matrix , binary_matrix_p_value, weighted_matrix_statistics = funcs.summarize_across_all_lags_v2_with_pvalues_more_directed_v2(graph, p_matrix_dec_4,statistics_dec_4, alpha_level)
            study_4_decoder_p_values_min[sub][ses][r] = binary_matrix_p_value
            study_4_decoder_statistics_max[sub][ses][r] = funcs.rearange_matrix_v2(weighted_matrix_statistics, network_parcel_idx)
            study4_decoder_binary_graph[sub][ses][r] = funcs.rearange_matrix_v2(binary_matrix, network_parcel_idx)
            study4_decoder_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(binary_matrix,net_labels, network_dic, parcel_labels)
            study4_decoder_partial_corr_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(weighted_matrix_statistics,net_labels, network_dic, parcel_labels)

            # study 5
            graph_dec_5 = multipledataset_calltif_decoder_study5_link_matrix[sub][ses][r]['graph']
            p_matrix_dec_5 = np.array(multipledataset_calltif_decoder_study5_link_matrix[sub][ses][r]['p_matrix'])
            statistics_dec_5 = np.array(multipledataset_calltif_decoder_study5_link_matrix[sub][ses][r]['val_matrix'])

            final_graph_new = p_matrix_dec_5 <= alpha_level
            graph = funcs.convert_to_string_graph(final_graph_new)
            study5_decoder_time_series_graph[sub][ses][r] = graph
            study5_decoder_time_series_graph_pvalues[sub][ses][r] = p_matrix_dec_5
            study5_decoder_time_series_graph_statistics[sub][ses][r] = statistics_dec_5

            binary_matrix , binary_matrix_p_value, weighted_matrix_statistics = funcs.summarize_across_all_lags_v2_with_pvalues_more_directed_v2(graph, p_matrix_dec_5,statistics_dec_5, alpha_level)
            study_5_decoder_p_values_min[sub][ses][r] = binary_matrix_p_value
            study_5_decoder_statistics_max[sub][ses][r] = funcs.rearange_matrix_v2(weighted_matrix_statistics, network_parcel_idx)
            study5_decoder_binary_graph[sub][ses][r] = funcs.rearange_matrix_v2(binary_matrix, network_parcel_idx)
            study5_decoder_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(binary_matrix,net_labels, network_dic, parcel_labels)
            study5_decoder_partial_corr_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(weighted_matrix_statistics,net_labels, network_dic, parcel_labels)

In [ ]:
all_studies_decoder_subnetworks = []
all_studies_decoder_partial_corr_subnetworks = []
all_studies_decoder_partial_correlations = []

for study in range(0, 5):
    for sub in range(0, num_subjects_ready):
         for ses in range(0, num_decoder_sessions):
            for r in range(num_dec_runs):
                if study == 0:
                    all_studies_decoder_subnetworks.append(np.array(study1_decoder_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_decoder_partial_corr_subnetworks.append(np.array(study1_decoder_partial_corr_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_decoder_partial_correlations.append(np.array(study_1_decoder_statistics_max)[sub,ses,r,:,:])
                elif study == 1:
                    all_studies_decoder_subnetworks.append(np.array(study2_decoder_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_decoder_partial_corr_subnetworks.append(np.array(study2_decoder_partial_corr_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_decoder_partial_correlations.append(np.array(study_2_decoder_statistics_max)[sub,ses,r,:,:])
                elif study == 2:
                    all_studies_decoder_subnetworks.append(np.array(study3_decoder_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_decoder_partial_corr_subnetworks.append(np.array(study3_decoder_partial_corr_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_decoder_partial_correlations.append(np.array(study_3_decoder_statistics_max)[sub,ses,r,:,:])
                elif study == 3:
                    all_studies_decoder_subnetworks.append(np.array(study4_decoder_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_decoder_partial_corr_subnetworks.append(np.array(study4_decoder_partial_corr_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_decoder_partial_correlations.append(np.array(study_4_decoder_statistics_max)[sub,ses,r,:,:])
                elif study == 4:
                    all_studies_decoder_subnetworks.append(np.array(study5_decoder_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_decoder_partial_corr_subnetworks.append(np.array(study5_decoder_partial_corr_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_decoder_partial_correlations.append(np.array(study_5_decoder_statistics_max)[sub,ses,r,:,:])

## Neurofeedback sessions

In [ ]:
study1_nf_time_series_graph =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study1_nf_time_series_graph_pvalues =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study1_nf_time_series_graph_statistics =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study1_nf_binary_graph =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study_1_nf_p_values_min = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study_1_nf_statistics_max = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study1_nf_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study1_nf_partial_corr_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 

study2_nf_time_series_graph =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study2_nf_time_series_graph_pvalues =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study2_nf_time_series_graph_statistics =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study2_nf_binary_graph =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study_2_nf_p_values_min = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study_2_nf_statistics_max = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study2_nf_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study2_nf_partial_corr_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 

study3_nf_time_series_graph =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study3_nf_time_series_graph_pvalues =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study3_nf_time_series_graph_statistics =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study3_nf_binary_graph = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study_3_nf_p_values_min = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study_3_nf_statistics_max = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study3_nf_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study3_nf_partial_corr_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 

study4_nf_time_series_graph =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study4_nf_time_series_graph_pvalues =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study4_nf_time_series_graph_statistics =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study4_nf_binary_graph =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study_4_nf_p_values_min = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study_4_nf_statistics_max = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study4_nf_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study4_nf_partial_corr_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 

study5_nf_time_series_graph =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study5_nf_time_series_graph_pvalues =  [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study5_nf_time_series_graph_statistics = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study5_nf_binary_graph = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study_5_nf_p_values_min = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study_5_nf_statistics_max = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study5_nf_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
study5_nf_partial_corr_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 

## nf sessions
for sub in range(0, num_subjects_ready):
    for ses in range(0, num_nf_sessions):
        for r in range(0, num_nf_runs):

            # study 1
            graph_nf_1 = multipledataset_calltif_nf_study1_link_matrix[sub][ses][r]['graph']
            p_matrix_nf_1 = np.array(multipledataset_calltif_nf_study1_link_matrix[sub][ses][r]['p_matrix'])
            statistics_nf_1 = np.array(multipledataset_calltif_nf_study1_link_matrix[sub][ses][r]['val_matrix'])

            final_graph_new = p_matrix_nf_1 <= alpha_level
            graph = funcs.convert_to_string_graph(final_graph_new)
            study1_nf_time_series_graph[sub][ses][r] = graph
            study1_nf_time_series_graph_pvalues[sub][ses][r] = p_matrix_nf_1
            study1_nf_time_series_graph_statistics[sub][ses][r] = statistics_nf_1
            
            binary_matrix , binary_matrix_p_value, weighted_matrix_statistics = funcs.summarize_across_all_lags_v2_with_pvalues_more_directed_v2(graph, p_matrix_nf_1,statistics_nf_1, alpha_level)
            study_1_nf_p_values_min[sub][ses][r] = binary_matrix_p_value
            study_1_nf_statistics_max[sub][ses][r] = funcs.rearange_matrix_v2(weighted_matrix_statistics,network_parcel_idx)
            study1_nf_binary_graph[sub][ses][r] = funcs.rearange_matrix_v2(binary_matrix, network_parcel_idx)
            study1_nf_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(binary_matrix,net_labels, network_dic, parcel_labels)
            study1_nf_partial_corr_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(weighted_matrix_statistics,net_labels, network_dic, parcel_labels)

            # study 2
            graph_nf_2 = multipledataset_calltif_nf_study2_link_matrix[sub][ses][r]['graph']
            p_matrix_nf_2 = np.array(multipledataset_calltif_nf_study2_link_matrix[sub][ses][r]['p_matrix'])
            statistics_nf_2 = np.array(multipledataset_calltif_nf_study2_link_matrix[sub][ses][r]['val_matrix'])

            final_graph_new = p_matrix_nf_2 <= alpha_level
            graph = funcs.convert_to_string_graph(final_graph_new)
            study2_nf_time_series_graph[sub][ses][r] = graph
            study2_nf_time_series_graph_pvalues[sub][ses][r] = p_matrix_nf_2
            study2_nf_time_series_graph_statistics[sub][ses][r] = statistics_nf_2

            binary_matrix , binary_matrix_p_value, weighted_matrix_statistics = funcs.summarize_across_all_lags_v2_with_pvalues_more_directed_v2(graph, p_matrix_nf_2,statistics_nf_2, alpha_level)
            study_2_nf_p_values_min[sub][ses][r] = binary_matrix_p_value
            study_2_nf_statistics_max[sub][ses][r] = funcs.rearange_matrix_v2(weighted_matrix_statistics,network_parcel_idx)
            study2_nf_binary_graph[sub][ses][r] = funcs.rearange_matrix_v2(binary_matrix, network_parcel_idx)
            study2_nf_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(binary_matrix,net_labels, network_dic, parcel_labels)
            study2_nf_partial_corr_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(weighted_matrix_statistics,net_labels, network_dic, parcel_labels)

            # study 3
            graph_nf_3 = multipledataset_calltif_nf_study3_link_matrix[sub][ses][r]['graph']
            p_matrix_nf_3 = np.array(multipledataset_calltif_nf_study3_link_matrix[sub][ses][r]['p_matrix'])
            statistics_nf_3 = np.array(multipledataset_calltif_nf_study3_link_matrix[sub][ses][r]['val_matrix'])

            final_graph_new = p_matrix_nf_3 <= alpha_level
            graph = funcs.convert_to_string_graph(final_graph_new)
            study3_nf_time_series_graph[sub][ses][r] = graph
            study3_nf_time_series_graph_pvalues[sub][ses][r] = p_matrix_nf_3
            study3_nf_time_series_graph_statistics[sub][ses][r] = statistics_nf_3

            binary_matrix , binary_matrix_p_value, weighted_matrix_statistics = funcs.summarize_across_all_lags_v2_with_pvalues_more_directed_v2(graph, p_matrix_nf_3,statistics_nf_3, alpha_level)
            study_3_nf_p_values_min[sub][ses][r] = binary_matrix_p_value
            study_3_nf_statistics_max[sub][ses][r] = funcs.rearange_matrix_v2(weighted_matrix_statistics,network_parcel_idx)
            study3_nf_binary_graph[sub][ses][r] = funcs.rearange_matrix_v2(binary_matrix, network_parcel_idx)
            study3_nf_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(binary_matrix,net_labels, network_dic, parcel_labels)
            study3_nf_partial_corr_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(weighted_matrix_statistics,net_labels, network_dic, parcel_labels)

            # study 4
            graph_nf_4 = multipledataset_calltif_nf_study4_link_matrix[sub][ses][r]['graph']
            p_matrix_nf_4 = np.array(multipledataset_calltif_nf_study4_link_matrix[sub][ses][r]['p_matrix'])
            statistics_nf_4 = np.array(multipledataset_calltif_nf_study4_link_matrix[sub][ses][r]['val_matrix'])

            final_graph_new = p_matrix_nf_4 <= alpha_level
            graph = funcs.convert_to_string_graph(final_graph_new)
            study4_nf_time_series_graph[sub][ses][r] = graph
            study4_nf_time_series_graph_pvalues[sub][ses][r] = p_matrix_nf_4
            study4_nf_time_series_graph_statistics[sub][ses][r] = statistics_nf_4

            binary_matrix , binary_matrix_p_value, weighted_matrix_statistics = funcs.summarize_across_all_lags_v2_with_pvalues_more_directed_v2(graph, p_matrix_nf_4,statistics_nf_4, alpha_level)
            study_4_nf_p_values_min[sub][ses][r] = binary_matrix_p_value
            study_4_nf_statistics_max[sub][ses][r] = funcs.rearange_matrix_v2(weighted_matrix_statistics,network_parcel_idx)
            study4_nf_binary_graph[sub][ses][r] = funcs.rearange_matrix_v2(binary_matrix, network_parcel_idx)
            study4_nf_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(binary_matrix,net_labels, network_dic, parcel_labels)
            study4_nf_partial_corr_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(weighted_matrix_statistics,net_labels, network_dic, parcel_labels)

            # study 5
            graph_nf_5 = multipledataset_calltif_nf_study5_link_matrix[sub][ses][r]['graph']
            p_matrix_nf_5 = np.array(multipledataset_calltif_nf_study5_link_matrix[sub][ses][r]['p_matrix'])
            statistics_nf_5 = np.array(multipledataset_calltif_nf_study5_link_matrix[sub][ses][r]['val_matrix'])

            final_graph_new = p_matrix_nf_5 <= alpha_level
            graph = funcs.convert_to_string_graph(final_graph_new)
            study5_nf_time_series_graph[sub][ses][r] = graph
            study5_nf_time_series_graph_pvalues[sub][ses][r] = p_matrix_nf_5
            study5_nf_time_series_graph_statistics[sub][ses][r] = statistics_nf_5

            binary_matrix , binary_matrix_p_value, weighted_matrix_statistics = funcs.summarize_across_all_lags_v2_with_pvalues_more_directed_v2(graph, p_matrix_nf_5,statistics_nf_5, alpha_level)
            study_5_nf_p_values_min[sub][ses][r]= binary_matrix_p_value
            study_5_nf_statistics_max[sub][ses][r] = funcs.rearange_matrix_v2(weighted_matrix_statistics,network_parcel_idx)
            study5_nf_binary_graph[sub][ses][r] = funcs.rearange_matrix_v2(binary_matrix, network_parcel_idx)
            study5_nf_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(binary_matrix,net_labels, network_dic, parcel_labels)
            study5_nf_partial_corr_subnetwork_graphs[sub][ses][r] = funcs.network_map_v2_without_decoder(weighted_matrix_statistics,net_labels, network_dic, parcel_labels)

In [ ]:
all_studies_nf_score_changes = []
all_studies_nf_score_mean = []
all_studies_nf_subnetworks = []
all_studies_nf_partial_corr_subnetworks = []
all_studies_nf_partial_corr = []

for study in range(0, 5):
    for sub in range(0, num_subjects_ready):
         for ses in range(0, num_nf_sessions):
            for r in range(num_nf_runs):
                if study == 0:
                    all_studies_nf_subnetworks.append(np.array(study1_nf_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_nf_partial_corr_subnetworks.append(np.array(study1_nf_partial_corr_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_nf_partial_corr.append(np.array(study_1_nf_statistics_max)[sub,ses,r,:,:])
                    all_studies_nf_score_changes.append(np.array(study1_scores_change)[sub, ses,r])
                    all_studies_nf_score_mean.append(np.array(study1_scores_avg)[sub, ses,r,0])

                elif study == 1:
                    all_studies_nf_subnetworks.append(np.array(study2_nf_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_nf_partial_corr_subnetworks.append(np.array(study2_nf_partial_corr_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_nf_partial_corr.append(np.array(study_2_nf_statistics_max)[sub,ses,r,:,:])
                    all_studies_nf_score_changes.append(np.array(study2_scores_change)[sub, ses,r])
                    all_studies_nf_score_mean.append(np.array(study2_scores_avg)[sub, ses,r,0])

                elif study == 2:
                    all_studies_nf_subnetworks.append(np.array(study3_nf_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_nf_partial_corr_subnetworks.append(np.array(study3_nf_partial_corr_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_nf_partial_corr.append(np.array(study_3_nf_statistics_max)[sub,ses,r,:,:])
                    all_studies_nf_score_changes.append(np.array(study3_scores_change)[sub, ses,r])
                    all_studies_nf_score_mean.append(np.array(study3_scores_avg)[sub, ses,r,0])

                elif study == 3:
                    all_studies_nf_subnetworks.append(np.array(study4_nf_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_nf_partial_corr_subnetworks.append(np.array(study4_nf_partial_corr_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_nf_partial_corr.append(np.array(study_4_nf_statistics_max)[sub,ses,r,:,:])
                    all_studies_nf_score_changes.append(np.array(study4_scores_change)[sub, ses,r])
                    all_studies_nf_score_mean.append(np.array(study4_scores_avg)[sub, ses,r,0])

                elif study == 4:
                    all_studies_nf_subnetworks.append(np.array(study5_nf_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_nf_partial_corr_subnetworks.append(np.array(study5_nf_partial_corr_subnetwork_graphs)[sub,ses,r,:,:])
                    all_studies_nf_partial_corr.append(np.array(study_5_nf_statistics_max)[sub,ses,r,:,:])
                    all_studies_nf_score_changes.append(np.array(study5_scores_change)[sub, ses,r])
                    all_studies_nf_score_mean.append(np.array(study5_scores_avg)[sub, ses,r,0])

# NF vs baseline (Figure 2, Supplementary Fig. 6)

## Edge-wise comparison of subnetwork-level graphs (partial correlations)

In [ ]:
NF_Dec_compare_edge_strength_statistics = np.zeros((num_networks, num_networks))
NF_Dec_compare_edge_strength_pvalue = np.zeros((num_networks, num_networks))

for i in range(0,num_networks):
    for j in range(0, num_networks):

        x = np.array(all_studies_nf_partial_corr_subnetworks)[:,i,j]
        y = np.array(all_studies_decoder_partial_corr_subnetworks)[:,i,j]

        temp = scipy.stats.ranksums(x, y, alternative='two-sided')
        NF_Dec_compare_edge_strength_statistics[i,j] = temp.statistic
        NF_Dec_compare_edge_strength_pvalue[i,j]= temp.pvalue

NF_Dec_compare_edge_strength_pvalue_corrected = np.reshape(statsmodels.stats.multitest.fdrcorrection(NF_Dec_compare_edge_strength_pvalue.flatten())[0], (num_networks, num_networks))

In [ ]:
plt.figure(figsize = (7,7))
ax1 = plt.subplot()
im = plt.imshow(np.multiply(NF_Dec_compare_edge_strength_statistics, NF_Dec_compare_edge_strength_pvalue_corrected), cmap = 'coolwarm', vmin = -np.max(np.abs(NF_Dec_compare_edge_strength_statistics)),  vmax = np.max(np.abs(NF_Dec_compare_edge_strength_statistics)))

plt.title('Significant Edges (NF-Baseline)', fontsize = 18)
ax1.set_xticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 10)
ax1.set_yticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 10)
g = plt.setp(ax1.get_xticklabels(), rotation =45, ha = "right",
        rotation_mode="anchor")

divider = make_axes_locatable(ax1)
cax = divider.append_axes("right", size="5%", pad=0.3)
clb = plt.colorbar(im, cax=cax)
clb.ax.tick_params(labelsize=8) 

plt.savefig(os.path.join(FIG_SUPP, 'SuppFig06c_partial_corr_NF_vs_baseline.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig06c_partial_corr_NF_vs_baseline.png'),bbox_inches='tight')

average_decoder_partial_corr_subnetworks = np.mean(np.array(all_studies_decoder_partial_corr_subnetworks), axis = 0)
average_nf_partial_corr_subnetworks = np.mean(np.array(all_studies_nf_partial_corr_subnetworks), axis = 0)

max_value = np.maximum(np.max(np.abs(average_decoder_partial_corr_subnetworks)),np.max(np.abs(average_nf_partial_corr_subnetworks)))

plt.figure(figsize = (7,7))
ax1 = plt.subplot()
im = plt.imshow(average_decoder_partial_corr_subnetworks, cmap = 'coolwarm', vmin = -max_value,  vmax = max_value)

plt.title('Average Causal Graph (Baseline)', fontsize = 18)
ax1.set_xticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 10)
ax1.set_yticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 10)
g = plt.setp(ax1.get_xticklabels(), rotation =45, ha = "right",
        rotation_mode="anchor")

divider = make_axes_locatable(ax1)
cax = divider.append_axes("right", size="5%", pad=0.3)
clb = plt.colorbar(im, cax=cax)
clb.ax.tick_params(labelsize=8) 

plt.savefig(os.path.join(FIG_SUPP, 'SuppFig06a_partial_corr_baseline.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig06a_partial_corr_baseline.png'),bbox_inches='tight')

plt.figure(figsize = (7,7))
ax1 = plt.subplot()
im = plt.imshow(average_nf_partial_corr_subnetworks, cmap = 'coolwarm', vmin = -max_value,  vmax =max_value)

plt.title('Average Causal Graph (NF)', fontsize = 18)
ax1.set_xticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 10)
ax1.set_yticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 10)
g = plt.setp(ax1.get_xticklabels(), rotation =45, ha = "right",
        rotation_mode="anchor")

divider = make_axes_locatable(ax1)
cax = divider.append_axes("right", size="5%", pad=0.3)
clb = plt.colorbar(im, cax=cax)
clb.ax.tick_params(labelsize=8) 

plt.savefig(os.path.join(FIG_SUPP, 'SuppFig06b_partial_corr_NF.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig06b_partial_corr_NF.png'),bbox_inches='tight')

In [ ]:
x = average_decoder_partial_corr_subnetworks
y = np.multiply(NF_Dec_compare_edge_strength_statistics, NF_Dec_compare_edge_strength_pvalue_corrected)

edges_positive_in_dc_and_more_positive_in_nf = np.zeros((num_networks, num_networks))
edges_positive_in_dc_and_less_positive_in_nf = np.zeros((num_networks, num_networks))
edges_negative_in_dc_and_more_negative_in_nf = np.zeros((num_networks, num_networks))
edges_negative_in_dc_and_less_negative_in_nf = np.zeros((num_networks, num_networks))

for i in range(0,num_networks):
     for j in range(0,num_networks):
        if x[i,j]>0 and y[i,j] >0:
            edges_positive_in_dc_and_more_positive_in_nf[i,j] = 1
        elif x[i,j]>0 and y[i,j] <0:
            edges_positive_in_dc_and_less_positive_in_nf[i,j] = 1
        elif x[i,j]<0 and y[i,j] <0:
            edges_negative_in_dc_and_more_negative_in_nf[i,j] = 1
        elif x[i,j]<0 and y[i,j] >0:
            edges_negative_in_dc_and_less_negative_in_nf[i,j] = 1

#DB4B46
#4696DB
#DB7846
#46DBD1

cmap = mcolors.ListedColormap(['white', '#DB4B46'])

plt.figure(figsize = (7,7))
ax1 = plt.subplot()
im = plt.imshow(edges_positive_in_dc_and_more_positive_in_nf, cmap = cmap)

plt.title('Positive at baseline,\nmore positive in NF', fontsize = 18, fontweight = 'bold')
ax1.set_xticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 12)
ax1.set_yticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 12)
g = plt.setp(ax1.get_xticklabels(), rotation =45, ha = "right",
        rotation_mode="anchor")

plt.savefig(os.path.join(FIG_MAIN, 'Fig2a_positive_more_positive_in_NF.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig2a_positive_more_positive_in_NF.png'),bbox_inches='tight')

cmap = mcolors.ListedColormap(['white', '#4696DB'])
plt.figure(figsize = (7,7))
ax1 = plt.subplot()
im = plt.imshow(edges_negative_in_dc_and_more_negative_in_nf, cmap = cmap)

plt.title('Negative at baseline,\nmore negative in NF', fontsize = 18,fontweight = 'bold')
ax1.set_xticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 12)
ax1.set_yticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 12)
g = plt.setp(ax1.get_xticklabels(), rotation =45, ha = "right",
        rotation_mode="anchor")

plt.savefig(os.path.join(FIG_MAIN, 'Fig2b_negative_more_negative_in_NF.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig2b_negative_more_negative_in_NF.png'),bbox_inches='tight')

cmap = mcolors.ListedColormap(['white', '#DB7846'])
plt.figure(figsize = (7,7))
ax1 = plt.subplot()
im = plt.imshow(edges_positive_in_dc_and_less_positive_in_nf, cmap = cmap)
plt.title('Positive at baseline,\nless positive in NF', fontsize = 18,fontweight = 'bold')
ax1.set_xticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 12)
ax1.set_yticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 12)
g = plt.setp(ax1.get_xticklabels(), rotation =45, ha = "right",
        rotation_mode="anchor")

plt.savefig(os.path.join(FIG_MAIN, 'Fig2c_positive_less_positive_in_NF.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig2c_positive_less_positive_in_NF.png'),bbox_inches='tight')

cmap = mcolors.ListedColormap(['white', '#46DBD1'])
plt.figure(figsize = (7,7))
ax1 = plt.subplot()
im = plt.imshow(edges_negative_in_dc_and_less_negative_in_nf, cmap = cmap)

plt.title('Negative at baseline,\nless negative in NF', fontsize = 18,fontweight = 'bold')
ax1.set_xticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 12)
ax1.set_yticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 12)
g = plt.setp(ax1.get_xticklabels(), rotation =45, ha = "right",
        rotation_mode="anchor")

plt.savefig(os.path.join(FIG_MAIN, 'Fig2d_negative_less_negative_in_NF.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig2d_negative_less_negative_in_NF.png'),bbox_inches='tight')

In [ ]:
visualization_labels
node_locs = np.array([[-15, -40, 10],[15, -40, 10], #sub
                      [-22, -90, 10],[22, -90, 10],  # Vis
                      [-35, -60, 10],[35, -60, 10], # Lim
                      [-10, 50, 10],[10, 50, 10],  # Cont
                      [-30, 30, 10],[30, 30, 10],  # DMN
                      [-40, -0, 10],[40, -0, 10], # Vent
                      [-10, -0, 10],[10, -0, 10], # Dors
                      [-35, -20, 10],[35, -20, 10], # Som-Mot
                      ])

node_positions_2d = node_locs[:, :2]

# Create a NetworkX graph from the adjacency matrix
G_positive_in_dc_and_more_positive_in_nf = nx.from_numpy_array(edges_positive_in_dc_and_more_positive_in_nf,create_using=nx.DiGraph)
G_positive_in_dc_and_less_positive_in_nf = nx.from_numpy_array(edges_positive_in_dc_and_less_positive_in_nf,create_using=nx.DiGraph)
G_negative_in_dc_and_more_negative_in_nf= nx.from_numpy_array(edges_negative_in_dc_and_more_negative_in_nf,create_using=nx.DiGraph)
G_negative_in_dc_and_less_negative_in_nf= nx.from_numpy_array(edges_negative_in_dc_and_less_negative_in_nf,create_using=nx.DiGraph)

node_size = 4000

# Plotting parameters
figure = plt.figure(figsize=(10, 10))
display_mode = 'z'

# Plot glass brain in orthographic view
display = plotting.plot_glass_brain(None, display_mode=display_mode,
                                    figure=figure, annotate=False, alpha=0.15)

node_positions = {i: (x, y) for i, (x, y) in enumerate(node_positions_2d)}

#DB4B46
#4696DB

nx.draw_networkx_edges(G_positive_in_dc_and_more_positive_in_nf, pos=node_positions, edge_color='#DB4B46', width=3, node_size=node_size, arrows=True, arrowsize=12)
nx.draw_networkx_edges(G_negative_in_dc_and_more_negative_in_nf, pos=node_positions, edge_color='#4696DB', width=3, node_size=node_size, arrows=True, arrowsize=12)

# # Draw nodes
nx.draw_networkx_nodes(G_positive_in_dc_and_more_positive_in_nf, pos=node_positions, node_color='lightgray', node_size=node_size)

node_labels_dict = {i: label for i, label in enumerate(short_label_modified)}
nx.draw_networkx_labels(G_positive_in_dc_and_more_positive_in_nf, pos=node_positions, labels=node_labels_dict, font_size=18, font_color='black', font_weight='bold')

plt.text(-15, 67, 'L', fontsize=26, ha='center', va='center', fontweight = 'bold')
plt.text(20, 67, 'R', fontsize=26, ha='center', va='center', fontweight = 'bold')

plt.savefig(os.path.join(FIG_MAIN, 'Fig2e_edges_stronger_in_NF.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig2e_edges_stronger_in_NF.png'),bbox_inches='tight')

# Plotting parameters
figure = plt.figure(figsize=(10, 10))

# Plot glass brain in orthographic view
display = plotting.plot_glass_brain(None, display_mode=display_mode,
                                    figure=figure, annotate=False, alpha=0.15)

node_positions = {i: (x, y) for i, (x, y) in enumerate(node_positions_2d)}

#DB7846
#46DBD1
nx.draw_networkx_edges(G_positive_in_dc_and_less_positive_in_nf, pos=node_positions, edge_color='#DB7846', width=3, node_size=node_size, arrows=True, arrowsize=12)
nx.draw_networkx_edges(G_negative_in_dc_and_less_negative_in_nf, pos=node_positions, edge_color='#46DBD1', width=3, node_size=node_size, arrows=True, arrowsize=12)

print(G_positive_in_dc_and_more_positive_in_nf.edges)
# # Draw nodes
nx.draw_networkx_nodes(G_positive_in_dc_and_more_positive_in_nf, pos=node_positions, node_color='lightgray', node_size=node_size)

node_labels_dict = {i: label for i, label in enumerate(short_label_modified)}
nx.draw_networkx_labels(G_positive_in_dc_and_more_positive_in_nf, pos=node_positions, labels=node_labels_dict, font_size=18, font_color='black', font_weight='bold')

plt.text(-15, 67, 'L', fontsize=26, ha='center', va='center', fontweight = 'bold')
plt.text(20, 67, 'R', fontsize=26, ha='center', va='center', fontweight = 'bold')

plt.savefig(os.path.join(FIG_MAIN, 'Fig2f_edges_weaker_in_NF.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig2f_edges_weaker_in_NF.png'),bbox_inches='tight')

### Degree and net flow (Fig. 2g,h)

In [ ]:
# Neurofeedback Sessions
study1_nf_subnetwork_graphs_degree = np.zeros((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks))
study1_nf_subnetwork_graphs_causal_flow =  np.zeros((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks))

study2_nf_subnetwork_graphs_degree =  np.zeros((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks))
study2_nf_subnetwork_graphs_causal_flow =  np.zeros((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks))

study3_nf_subnetwork_graphs_degree =  np.zeros((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks))
study3_nf_subnetwork_graphs_causal_flow =  np.zeros((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks))

study4_nf_subnetwork_graphs_degree =   np.zeros((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks))
study4_nf_subnetwork_graphs_causal_flow =  np.zeros((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks))

study5_nf_subnetwork_graphs_degree =   np.zeros((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks))
study5_nf_subnetwork_graphs_causal_flow =  np.zeros((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks))

k =0
for sub in range(0, num_subjects_ready):
    for ses in range(0, num_nf_sessions):
        for r in range(0, num_nf_runs):

            study1_nf_subnetwork_graphs_degree[k,:] = np.mean(study1_nf_subnetwork_graphs[sub][ses][r], axis = 1)+np.mean(study1_nf_subnetwork_graphs[sub][ses][r], axis = 0)
            study1_nf_subnetwork_graphs_causal_flow[k,:] = np.mean(study1_nf_subnetwork_graphs[sub][ses][r], axis = 1)-np.mean(study1_nf_subnetwork_graphs[sub][ses][r], axis = 0)

            study2_nf_subnetwork_graphs_degree[k,:] = np.mean(study2_nf_subnetwork_graphs[sub][ses][r], axis = 1)+np.mean(study2_nf_subnetwork_graphs[sub][ses][r], axis = 0)
            study2_nf_subnetwork_graphs_causal_flow[k,:] = np.mean(study2_nf_subnetwork_graphs[sub][ses][r], axis = 1)-np.mean(study2_nf_subnetwork_graphs[sub][ses][r], axis = 0)

            study3_nf_subnetwork_graphs_degree[k,:] = np.mean(study3_nf_subnetwork_graphs[sub][ses][r], axis = 1)+np.mean(study3_nf_subnetwork_graphs[sub][ses][r], axis = 0)
            study3_nf_subnetwork_graphs_causal_flow[k,:] = np.mean(study3_nf_subnetwork_graphs[sub][ses][r], axis = 1)-np.mean(study3_nf_subnetwork_graphs[sub][ses][r], axis = 0)

            study4_nf_subnetwork_graphs_degree[k,:] = np.mean(study4_nf_subnetwork_graphs[sub][ses][r], axis = 1)+np.mean(study4_nf_subnetwork_graphs[sub][ses][r], axis = 0)
            study4_nf_subnetwork_graphs_causal_flow[k,:] = np.mean(study4_nf_subnetwork_graphs[sub][ses][r], axis = 1)-np.mean(study4_nf_subnetwork_graphs[sub][ses][r], axis = 0)

            study5_nf_subnetwork_graphs_degree[k,:] = np.mean(study5_nf_subnetwork_graphs[sub][ses][r], axis = 1)+np.mean(study5_nf_subnetwork_graphs[sub][ses][r], axis = 0)
            study5_nf_subnetwork_graphs_causal_flow[k,:] = np.mean(study5_nf_subnetwork_graphs[sub][ses][r], axis = 1)-np.mean(study5_nf_subnetwork_graphs[sub][ses][r], axis = 0)

            k = k + 1

# Decoder sessions

study1_decoder_subnetwork_graphs_degree =  np.zeros((num_subjects_ready*num_dec_runs*num_decoder_sessions,num_networks))
study1_decoder_subnetwork_graphs_causal_flow =  np.zeros((num_subjects_ready*num_dec_runs*num_decoder_sessions,num_networks))

study2_decoder_subnetwork_graphs_degree =  np.zeros((num_subjects_ready*num_dec_runs*num_decoder_sessions,num_networks))
study2_decoder_subnetwork_graphs_causal_flow =  np.zeros((num_subjects_ready*num_dec_runs*num_decoder_sessions,num_networks))

study3_decoder_subnetwork_graphs_degree =  np.zeros((num_subjects_ready*num_dec_runs*num_decoder_sessions,num_networks))
study3_decoder_subnetwork_graphs_causal_flow =  np.zeros((num_subjects_ready*num_dec_runs*num_decoder_sessions,num_networks))

study4_decoder_subnetwork_graphs_degree =   np.zeros((num_subjects_ready*num_dec_runs*num_decoder_sessions,num_networks))
study4_decoder_subnetwork_graphs_causal_flow =  np.zeros((num_subjects_ready*num_dec_runs*num_decoder_sessions,num_networks))

study5_decoder_subnetwork_graphs_degree =   np.zeros((num_subjects_ready*num_dec_runs*num_decoder_sessions,num_networks))
study5_decoder_subnetwork_graphs_causal_flow =  np.zeros((num_subjects_ready*num_dec_runs*num_decoder_sessions,num_networks))

k =0
for sub in range(0, num_subjects_ready):
    for ses in range(0, num_decoder_sessions):
        for r in range(0,num_dec_runs):

            study1_decoder_subnetwork_graphs_degree[k,:] = np.mean(study1_decoder_subnetwork_graphs[sub][ses][r], axis = 1)+np.mean(study1_decoder_subnetwork_graphs[sub][ses][r], axis = 0)
            study1_decoder_subnetwork_graphs_causal_flow[k,:] = np.mean(study1_decoder_subnetwork_graphs[sub][ses][r], axis = 1)-np.mean(study1_decoder_subnetwork_graphs[sub][ses][r], axis = 0)

            study2_decoder_subnetwork_graphs_degree[k,:] = np.mean(study2_decoder_subnetwork_graphs[sub][ses][r], axis = 1)+np.mean(study2_decoder_subnetwork_graphs[sub][ses][r], axis = 0)
            study2_decoder_subnetwork_graphs_causal_flow[k,:] = np.mean(study2_decoder_subnetwork_graphs[sub][ses][r], axis = 1)-np.mean(study2_decoder_subnetwork_graphs[sub][ses][r], axis = 0)

            study3_decoder_subnetwork_graphs_degree[k,:] = np.mean(study3_decoder_subnetwork_graphs[sub][ses][r], axis = 1)+np.mean(study3_decoder_subnetwork_graphs[sub][ses][r], axis = 0)
            study3_decoder_subnetwork_graphs_causal_flow[k,:] = np.mean(study3_decoder_subnetwork_graphs[sub][ses][r], axis = 1)-np.mean(study3_decoder_subnetwork_graphs[sub][ses][r], axis = 0)

            study4_decoder_subnetwork_graphs_degree[k,:] = np.mean(study4_decoder_subnetwork_graphs[sub][ses][r], axis = 1)+np.mean(study4_decoder_subnetwork_graphs[sub][ses][r], axis = 0)
            study4_decoder_subnetwork_graphs_causal_flow[k,:] = np.mean(study4_decoder_subnetwork_graphs[sub][ses][r], axis = 1)-np.mean(study4_decoder_subnetwork_graphs[sub][ses][r], axis = 0)

            study5_decoder_subnetwork_graphs_degree[k,:] = np.mean(study5_decoder_subnetwork_graphs[sub][ses][r], axis = 1)+np.mean(study5_decoder_subnetwork_graphs[sub][ses][r], axis = 0)
            study5_decoder_subnetwork_graphs_causal_flow[k,:] = np.mean(study5_decoder_subnetwork_graphs[sub][ses][r], axis = 1)-np.mean(study5_decoder_subnetwork_graphs[sub][ses][r], axis = 0)

            k = k + 1

In [ ]:
# degree
all_studies_nf_degrees = np.concatenate((study1_nf_subnetwork_graphs_degree,study2_nf_subnetwork_graphs_degree, study3_nf_subnetwork_graphs_degree, study4_nf_subnetwork_graphs_degree,study5_nf_subnetwork_graphs_degree), axis = 0)
all_studies_decoder_degrees = np.concatenate((study1_decoder_subnetwork_graphs_degree,study2_decoder_subnetwork_graphs_degree, study3_decoder_subnetwork_graphs_degree, study4_decoder_subnetwork_graphs_degree,study5_decoder_subnetwork_graphs_degree), axis = 0)
print(all_studies_nf_degrees.shape,all_studies_decoder_degrees.shape)

# causal flow
all_studies_nf_causal_flow = np.concatenate((study1_nf_subnetwork_graphs_causal_flow,study2_nf_subnetwork_graphs_causal_flow, study3_nf_subnetwork_graphs_causal_flow, study4_nf_subnetwork_graphs_causal_flow,study5_nf_subnetwork_graphs_causal_flow), axis = 0)
all_studies_decoder_causal_flow = np.concatenate((study1_decoder_subnetwork_graphs_causal_flow,study2_decoder_subnetwork_graphs_causal_flow, study3_decoder_subnetwork_graphs_causal_flow, study4_decoder_subnetwork_graphs_causal_flow,study5_decoder_subnetwork_graphs_causal_flow), axis = 0)
print(all_studies_nf_causal_flow.shape,all_studies_decoder_causal_flow.shape)

sub_net_all_studies_degree_diff_nf_dec_pvalue = []
sub_net_all_studies_degree_diff_nf_dec_statistics = []

sub_net_all_studies_causal_flow_diff_nf_dec_pvalue = []
sub_net_all_studies_causal_flow_diff_nf_dec_statistics = []

for n in range(0,num_networks):

    #degree
    temp = scipy.stats.ranksums(all_studies_nf_degrees[:,n], all_studies_decoder_degrees[:,n], alternative='two-sided')
    sub_net_all_studies_degree_diff_nf_dec_statistics.append(temp.statistic)
    sub_net_all_studies_degree_diff_nf_dec_pvalue.append(temp.pvalue)

    # causal flow
    temp = scipy.stats.ranksums(all_studies_nf_causal_flow[:,n], all_studies_decoder_causal_flow[:,n], alternative='two-sided')
    sub_net_all_studies_causal_flow_diff_nf_dec_statistics.append(temp.statistic)
    sub_net_all_studies_causal_flow_diff_nf_dec_pvalue.append(temp.pvalue)

# degree 
sub_net_all_studies_degree_diff_nf_dec_pvalue_corrected = statsmodels.stats.multitest.fdrcorrection(sub_net_all_studies_degree_diff_nf_dec_pvalue)[0]
print('Significantly Higher Degree in NF than in DC:',visualization_labels[np.where(sub_net_all_studies_degree_diff_nf_dec_pvalue_corrected == True)[0][0]], sub_net_all_studies_degree_diff_nf_dec_statistics[np.where(sub_net_all_studies_degree_diff_nf_dec_pvalue_corrected == True)[0][0]])
print(statsmodels.stats.multitest.fdrcorrection(sub_net_all_studies_degree_diff_nf_dec_pvalue))
# pvalue = 2 star
# causal flow 
sub_net_all_studies_causal_flow_diff_nf_dec_pvalue_corrected = statsmodels.stats.multitest.fdrcorrection(sub_net_all_studies_causal_flow_diff_nf_dec_pvalue)[0]
print('Significantly Higher Causal Flow in NF than in DC:',visualization_labels[np.where(sub_net_all_studies_causal_flow_diff_nf_dec_pvalue_corrected == True)[0][0]], sub_net_all_studies_causal_flow_diff_nf_dec_statistics[np.where(sub_net_all_studies_causal_flow_diff_nf_dec_pvalue_corrected == True)[0][0]])
print(statsmodels.stats.multitest.fdrcorrection(sub_net_all_studies_causal_flow_diff_nf_dec_pvalue))
# pvalue: one star

In [ ]:
data1_long = pd.DataFrame(all_studies_decoder_degrees, columns=visualization_labels).melt( value_name='Value')
data1_long['Type'] = 'Baseline'
data2_long = pd.DataFrame(all_studies_nf_degrees, columns=visualization_labels).melt(value_name='Value')
data2_long['Type'] = 'NF'

y_label_font = 38
y_ticks_font = 24
x_ticks_font = 26
legend_font = 22
significance_font = 16

combined_data = pd.concat([data1_long, data2_long])

# Plot
plt.figure(figsize=(12, 6))
ax =sns.barplot(x='variable', y='Value', hue='Type', data=combined_data, palette=['lightgreen', 'green'], errorbar= 'se')

# Customize labels and title
plt.ylabel('Nodal Degree', fontsize = y_label_font)
plt.grid(True)
plt.xticks(rotation=65, fontsize = x_ticks_font, ha = 'center')
plt.yticks(fontsize = y_ticks_font)
plt.xlabel('')
x_left, x_right = plt.xlim()
plt.xlim(-0.5, x_right)

#L-Sub
x1, x2 = -0.3, 0.3
y, h, col = 0.4, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

#R-Sub
x1, x2 = 0.7, 1.3
y, h, col = 0.4, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

#L-vis
x1, x2 = 1.7, 2.3
y, h, col = 0.5, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)
# R-vis
x1, x2 = 2.7, 3.3
y, h, col = 0.5, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

#L-lim
x1, x2 = 3.7, 4.3
y, h, col = 0.45, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col,fontweight = 'bold', fontsize = significance_font)
# R-lim
x1, x2 = 4.7, 5.3
y, h, col = 0.45, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "**", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)
#L-cont
x1, x2 = 5.7, 6.3
y, h, col = 0.5, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)
#R-cont
x1, x2 = 6.7, 7.3
y, h, col = 0.52, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)
#L-DMN
x1, x2 = 7.7, 8.3
y, h, col = 0.52, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)
#R-DMN
x1, x2 = 8.7, 9.3
y, h, col = 0.52, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)
#L-vent-att
x1, x2 = 9.7, 10.3
y, h, col = 0.62, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col,fontweight = 'bold', fontsize = significance_font)

#R-vent-att
x1, x2 = 10.7, 11.3
y, h, col = 0.68, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

#L-Dors-att
x1, x2 = 11.7, 12.3
y, h, col = 0.62, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col,fontweight = 'bold', fontsize = significance_font)
#R-Dors-att
x1, x2 = 12.7, 13.3
y, h, col = 0.62, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

#L-Som-Mot
x1, x2 = 13.7, 14.3
y, h, col = 0.65, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)
#R-Som-Mot
x1, x2 = 14.7, 15.3
y, h, col = 0.65, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

plt.legend(loc='upper left', fontsize = legend_font, bbox_to_anchor=(-0.01, 1.05), framealpha = 0)

plt.tight_layout()
plt.savefig(os.path.join(FIG_MAIN, 'Fig2g_degree.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig2g_degree.png'),bbox_inches='tight')

In [ ]:
data1_long = pd.DataFrame(all_studies_decoder_causal_flow, columns=visualization_labels).melt( value_name='Value')
data1_long['Type'] = 'Baseline'
data2_long = pd.DataFrame(all_studies_nf_causal_flow, columns=visualization_labels).melt(value_name='Value')
data2_long['Type'] = 'NF'

combined_data = pd.concat([data1_long, data2_long])

# Plot
plt.figure(figsize=(12, 6))
ax = sns.barplot(x='variable', y='Value', hue='Type', data=combined_data, palette=['lightgreen', 'green'],  errorbar= 'se')

# Customize labels and title
plt.ylabel('Net Flow', fontsize = y_label_font)
plt.grid(True)
plt.xticks(rotation=65, fontsize = x_ticks_font)
plt.yticks(fontsize = y_ticks_font)
plt.xlabel('')

# sub
x1, x2 = -0.3, 0.3
y, h, col = 0.001, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 0.7, 1.3
y, h, col = 0.001, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

# vis
x1, x2 = 1.7, 2.3
y, h, col = 0.001, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 2.7, 3.3
y, h, col = -0.001, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-5*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

# Lim

x1, x2 = 3.7, 4.3
y, h, col =0.0028, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 4.7, 5.3
y, h, col = -0.001, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-5*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

# cont

x1, x2 = 5.7, 6.3
y, h, col =0.0017, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 6.7, 7.3
y, h, col = -0.001, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-5*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

#dmn

x1, x2 = 7.7, 8.3
y, h, col =-0.0015, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-5*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 8.7, 9.3
y, h, col =-0.0015, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-5*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

#vent-att

x1, x2 = 9.7, 10.3
y, h, col = 0.002, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 10.7, 11.3
y, h, col =-0.0045, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-5.5*h, "*", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

# dors-att

x1, x2 = 11.7, 12.3
y, h, col = 0.0024, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 12.7, 13.3
y, h, col =-0.001, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-5*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

# som-mot
x1, x2 = 13.7, 14.3
y, h, col = 0.002, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 14.7, 15.3
y, h, col =-0.0014, 0.00015, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-5*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x_left, x_right = plt.xlim()
plt.xlim(-0.5, x_right-0.5)

plt.legend(loc='lower left', fontsize = legend_font, bbox_to_anchor=(-0.01, 0.0), framealpha = 0)

plt.tight_layout()
plt.savefig(os.path.join(FIG_MAIN, 'Fig2h_net_flow.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig2h_net_flow.png'),bbox_inches='tight')

### Global network measures, NF vs baseline (Fig. 2i)

In [ ]:
# Neurofeedback Sessions
all_studies_nf_subnetwork_graphs_density = []
all_studies_nf_subnetwork_graphs_shortest_path_length =  []
all_studies_nf_subnetwork_graphs_assortativity = []
all_studies_nf_subnetwork_graphs_modularity = []
all_studies_nf_subnetwork_graphs_spectral_radius = []
all_studies_nf_subnetwork_graphs_synchronizability = []

for s in range(0,np.array(all_studies_nf_subnetworks).shape[0]):

    weighted_matrix = np.array(all_studies_nf_subnetworks)[s,:,:]
    G = funcs.create_weighted_digraph_from_matrix(weighted_matrix)

    all_studies_nf_subnetwork_graphs_density.append(np.mean(weighted_matrix))
    
    all_studies_nf_subnetwork_graphs_shortest_path_length.append(nx.average_shortest_path_length(G))
    
    all_studies_nf_subnetwork_graphs_assortativity.append(nx.degree_assortativity_coefficient(G, weight='weight'))
    
    G_undirected = G.to_undirected()
    partition = nx.community.greedy_modularity_communities(G_undirected, weight='weight')
    modularity = nx.community.modularity(G_undirected, partition, weight='weight')
    
    all_studies_nf_subnetwork_graphs_modularity.append(modularity)
    
    eigenvalues = np.linalg.eigvals(weighted_matrix)
    max_abs_eigenvalue = max(abs(eigenvalues))
    all_studies_nf_subnetwork_graphs_spectral_radius.append(max_abs_eigenvalue)

    L = nx.linalg.laplacianmatrix.directed_laplacian_matrix(G, weight='weight')
    eigenvalues, _ = np.linalg.eig(L)
    eigenvalues_sorted = np.sort(eigenvalues)
    lambda_L2 = eigenvalues_sorted[1]  # Second smallest eigenvalue
    lambda_Lmax = eigenvalues_sorted[-1]  # Largest eigenvalue
    S = lambda_L2 / lambda_Lmax
    all_studies_nf_subnetwork_graphs_synchronizability.append(S)

In [ ]:
# Neurofeedback Sessions
all_studies_decoder_subnetwork_graphs_density = []
all_studies_decoder_subnetwork_graphs_shortest_path_length =  []
all_studies_decoder_subnetwork_graphs_assortativity = []
all_studies_decoder_subnetwork_graphs_modularity = []
all_studies_decoder_subnetwork_graphs_spectral_radius = []
all_studies_decoder_subnetwork_graphs_synchronizability = []

for s in range(0,np.array(all_studies_decoder_subnetworks).shape[0]):

    weighted_matrix = np.array(all_studies_decoder_subnetworks)[s,:,:]
    G = funcs.create_weighted_digraph_from_matrix(weighted_matrix)

    all_studies_decoder_subnetwork_graphs_density.append(np.mean(weighted_matrix))
    
    all_studies_decoder_subnetwork_graphs_shortest_path_length.append(nx.average_shortest_path_length(G))
    
    all_studies_decoder_subnetwork_graphs_assortativity.append(nx.degree_assortativity_coefficient(G, weight='weight'))
    
    G_undirected = G.to_undirected()
    partition = nx.community.greedy_modularity_communities(G_undirected, weight='weight')
    modularity = nx.community.modularity(G_undirected, partition, weight='weight')
    
    all_studies_decoder_subnetwork_graphs_modularity.append(modularity)
    
    eigenvalues = np.linalg.eigvals(weighted_matrix)
    max_abs_eigenvalue = max(abs(eigenvalues))
    all_studies_decoder_subnetwork_graphs_spectral_radius.append(max_abs_eigenvalue)

    L = nx.linalg.laplacianmatrix.directed_laplacian_matrix(G, weight='weight')
    eigenvalues, _ = np.linalg.eig(L)
    eigenvalues_sorted = np.sort(eigenvalues)
    lambda_L2 = eigenvalues_sorted[1]  # Second smallest eigenvalue
    lambda_Lmax = eigenvalues_sorted[-1]  # Largest eigenvalue
    S = lambda_L2 / lambda_Lmax
    all_studies_decoder_subnetwork_graphs_synchronizability.append(S)

In [ ]:
global_measures_labels= ['Density','Shortest \n Path Length','Assortativity','Modularity','Spectral \n Radius','Synchroniz-\nability']

all_studies_all_global_measures_nf = np.vstack((all_studies_nf_subnetwork_graphs_density,all_studies_nf_subnetwork_graphs_shortest_path_length,all_studies_nf_subnetwork_graphs_assortativity,all_studies_nf_subnetwork_graphs_modularity,all_studies_nf_subnetwork_graphs_spectral_radius,all_studies_nf_subnetwork_graphs_synchronizability)).T
print(all_studies_all_global_measures_nf.shape)

all_studies_all_global_measures_decoder = np.vstack((all_studies_decoder_subnetwork_graphs_density,all_studies_decoder_subnetwork_graphs_shortest_path_length,all_studies_decoder_subnetwork_graphs_assortativity,all_studies_decoder_subnetwork_graphs_modularity,all_studies_decoder_subnetwork_graphs_spectral_radius,all_studies_decoder_subnetwork_graphs_synchronizability)).T
print(all_studies_all_global_measures_decoder.shape)

data1_long = pd.DataFrame(all_studies_all_global_measures_decoder, columns=global_measures_labels).melt( value_name='Value')
data1_long['Type'] = 'Baseline'
data2_long = pd.DataFrame(all_studies_all_global_measures_nf, columns=global_measures_labels).melt(value_name='Value')
data2_long['Type'] = 'NF'

combined_data = pd.concat([data1_long, data2_long])

# Plot
plt.figure(figsize=(12, 6))
ax = sns.barplot(x='variable', y='Value', hue='Type', data=combined_data, palette=['lightgreen', 'green'], errorbar= 'se', width =0.4 )

# Customize labels and title
plt.grid(True)
plt.xticks(rotation=45, fontsize = x_ticks_font)
plt.yticks(fontsize = y_ticks_font)

plt.xlabel('')
plt.ylabel('')
plt.yscale('log')

x1, x2 = -0.1, 0.1
y, h, col =0.3, 0.015, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 0.9, 1.1
y, h, col =1.3, 0.03, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "**", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 1.9, 2.1
y, h, col =0.02, 0.001, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 2.9, 3.1
y, h, col =0.3, 0.015, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 3.9, 4.1
y, h, col =5, 0.3, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 4.9, 5.1
y, h, col =0.6, 0.03, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x_left, x_right = plt.xlim()
plt.xlim(-0.3, x_right-0.2)

plt.legend(loc='upper center', fontsize = legend_font, framealpha = 0)

plt.tight_layout()
plt.savefig(os.path.join(FIG_MAIN, 'Fig2i_global_measures.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig2i_global_measures.png'),bbox_inches='tight')

In [ ]:
# Pairwise Pearson similarity between all NF graphs and between all baseline graphs (Fig. 1d).
# Computed once and cached in Results/, since it takes a while.
SIM_CACHE = 'Results/nf_dc_similarities_distributions.npz'
if os.path.exists(SIM_CACHE):
    import_npz(SIM_CACHE)
else:
    def pairwise_similarity(graphs):
        g = np.array(graphs).reshape(len(graphs), -1)
        return np.corrcoef(g)[np.triu_indices(len(g), k=1)]
    similarities_between_NF_graphs_dist = pairwise_similarity(all_studies_nf_subnetworks)
    similarities_between_DC_graphs_dist = pairwise_similarity(all_studies_decoder_subnetworks)
    np.savez(SIM_CACHE, similarities_between_NF_graphs_dist=similarities_between_NF_graphs_dist,
             similarities_between_DC_graphs_dist=similarities_between_DC_graphs_dist)

In [ ]:
plt.figure(figsize = (8,8))

data = pd.DataFrame({'Value': np.concatenate([similarities_between_DC_graphs_dist, similarities_between_NF_graphs_dist]),
                     'Session': ['Baseline'] * len(similarities_between_DC_graphs_dist) + ['NF'] * len(similarities_between_NF_graphs_dist),
                     'Category': [''] * (len(similarities_between_NF_graphs_dist) + len(similarities_between_DC_graphs_dist))})  # Single category

# Plot half-violin plots with split parameter
ax = sns.violinplot(x='Category', y='Value', hue='Session', data=data, palette=['lightgreen', 'green'], split=True, linewidth=2, inner="quartile")

for i, artist in enumerate(plt.gca().artists):
    artist.set_alpha(0.1)  # Transparency
    artist.set_linewidth(4)  # Border width

    if i % 2 == 0:  # Left half
        artist.set_clip_box(plt.gca().bbox)
        artist.set_width(0.1)  # Adjust the width
    else:  # Right half
        artist.set_clip_box(plt.gca().bbox)
        artist.set_width(0.1)  # Adjust the width
plt.xlabel('')

for artist in ax.lines:
    artist.set_linestyle('--')  # Change line style (optional)
    artist.set_linewidth(2)     # Change line width
    artist.set_color('gray')     # Change line color

plt.ylabel('Causal Graph Similarity', fontsize = y_label_font)
plt.yticks(fontsize = y_ticks_font)
plt.legend(fontsize = legend_font, framealpha=0,loc = 'lower left' , bbox_to_anchor=(-0.04, -0.04))

plt.text(0.25, 0.2, r'$p < 10^{-140}$ ', fontsize=30, ha='center', va='center', color = 'black')

plt.savefig(os.path.join(FIG_MAIN, 'Fig1d_similarity_NF_vs_baseline.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig1d_similarity_NF_vs_baseline.png'),bbox_inches='tight')

print(stats.ranksums(similarities_between_DC_graphs_dist, similarities_between_NF_graphs_dist, alternative= 'greater'))

## Graph similarity (Fig. 1c,d)

### The distribution of similarities between NFs of each subject (this is a distribution of 21choose2x9x5 samples total)

In [ ]:
study1_nf_subnetwork_graphs_each_subject = np.array(study1_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))
study2_nf_subnetwork_graphs_each_subject = np.array(study2_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))
study3_nf_subnetwork_graphs_each_subject = np.array(study3_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))
study4_nf_subnetwork_graphs_each_subject = np.array(study4_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))
study5_nf_subnetwork_graphs_each_subject = np.array(study5_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))

corr_study1 = np.zeros((num_nf_sessions*num_nf_runs, num_nf_sessions*num_nf_runs))
corr_study2 = np.zeros((num_nf_sessions*num_nf_runs, num_nf_sessions*num_nf_runs))
corr_study3 = np.zeros((num_nf_sessions*num_nf_runs, num_nf_sessions*num_nf_runs))
corr_study4 = np.zeros((num_nf_sessions*num_nf_runs, num_nf_sessions*num_nf_runs))
corr_study5 = np.zeros((num_nf_sessions*num_nf_runs, num_nf_sessions*num_nf_runs))

similarities_between_NF_graphs_of_each_subject_dist= []
for sub in range(0,num_subjects_ready):
    for k1 in range(0, num_nf_sessions*num_nf_runs):
        for k2 in range(0, num_nf_sessions*num_nf_runs):
           
           # study 1
            graph1 = study1_nf_subnetwork_graphs_each_subject[sub,k1]
            graph2 = study1_nf_subnetwork_graphs_each_subject[sub,k2]
            corr_study1[k1,k2] = stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic

          # study 2
            graph1 = study2_nf_subnetwork_graphs_each_subject[sub,k1]
            graph2 = study2_nf_subnetwork_graphs_each_subject[sub,k2]
            corr_study2[k1,k2] = stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic

         # study 3
            graph1 = study3_nf_subnetwork_graphs_each_subject[sub,k1]
            graph2 = study3_nf_subnetwork_graphs_each_subject[sub,k2]
            corr_study3[k1,k2] = stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic

        # study 4
            graph1 = study4_nf_subnetwork_graphs_each_subject[sub,k1]
            graph2 = study4_nf_subnetwork_graphs_each_subject[sub,k2]
            corr_study4[k1,k2] = stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic

        # study 5
            graph1 = study5_nf_subnetwork_graphs_each_subject[sub,k1]
            graph2 = study5_nf_subnetwork_graphs_each_subject[sub,k2]
            corr_study5[k1,k2] = stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic

    similarities_between_NF_graphs_of_each_subject_dist.append(corr_study1[np.triu_indices(num_nf_sessions*num_nf_runs, k= 1)])
    similarities_between_NF_graphs_of_each_subject_dist.append(corr_study2[np.triu_indices(num_nf_sessions*num_nf_runs, k= 1)])
    similarities_between_NF_graphs_of_each_subject_dist.append(corr_study3[np.triu_indices(num_nf_sessions*num_nf_runs, k= 1)])
    similarities_between_NF_graphs_of_each_subject_dist.append(corr_study4[np.triu_indices(num_nf_sessions*num_nf_runs, k= 1)])
    similarities_between_NF_graphs_of_each_subject_dist.append(corr_study5[np.triu_indices(num_nf_sessions*num_nf_runs, k= 1)])

### The distribution of similarities b/w NF and DC of each subject (this is a distribution of 21x5 samples per each subject, which makes the total sample count = 21x5x9x5)

In [ ]:
study1_nf_subnetwork_graphs_each_subject = np.array(study1_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))
study2_nf_subnetwork_graphs_each_subject = np.array(study2_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))
study3_nf_subnetwork_graphs_each_subject = np.array(study3_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))
study4_nf_subnetwork_graphs_each_subject = np.array(study4_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))
study5_nf_subnetwork_graphs_each_subject = np.array(study5_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))

study1_dec_subnetwork_graphs_each_subject = np.array(study1_decoder_subnetwork_graphs).reshape((num_subjects_ready,num_decoder_sessions*num_dec_runs,num_networks, num_networks))
study2_dec_subnetwork_graphs_each_subject = np.array(study2_decoder_subnetwork_graphs).reshape((num_subjects_ready,num_decoder_sessions*num_dec_runs,num_networks, num_networks))
study3_dec_subnetwork_graphs_each_subject = np.array(study3_decoder_subnetwork_graphs).reshape((num_subjects_ready,num_decoder_sessions*num_dec_runs,num_networks, num_networks))
study4_dec_subnetwork_graphs_each_subject = np.array(study4_decoder_subnetwork_graphs).reshape((num_subjects_ready,num_decoder_sessions*num_dec_runs,num_networks, num_networks))
study5_dec_subnetwork_graphs_each_subject = np.array(study5_decoder_subnetwork_graphs).reshape((num_subjects_ready,num_decoder_sessions*num_dec_runs,num_networks, num_networks))

similarities_between_DC_and_NF_graphs_of_each_subject_dist= []
for sub in range(0,num_subjects_ready):
    for k1 in range(0, num_nf_sessions*num_nf_runs):
        for k2 in range(0, num_decoder_sessions*num_dec_runs):
           
           # study 1
            graph1 = study1_nf_subnetwork_graphs_each_subject[sub][k1]
            graph2 = study1_dec_subnetwork_graphs_each_subject[sub,k2]
            similarities_between_DC_and_NF_graphs_of_each_subject_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)

          # study 2
            graph1 = study2_nf_subnetwork_graphs_each_subject[sub,k1]
            graph2 = study2_dec_subnetwork_graphs_each_subject[sub,k2]
            similarities_between_DC_and_NF_graphs_of_each_subject_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)

         # study 3
            graph1 = study3_nf_subnetwork_graphs_each_subject[sub,k1]
            graph2 = study3_dec_subnetwork_graphs_each_subject[sub,k2]
            similarities_between_DC_and_NF_graphs_of_each_subject_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)

        # study 4
            graph1 = study4_nf_subnetwork_graphs_each_subject[sub,k1]
            graph2 = study4_dec_subnetwork_graphs_each_subject[sub,k2]
            similarities_between_DC_and_NF_graphs_of_each_subject_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)

        # study 5
            graph1 = study5_nf_subnetwork_graphs_each_subject[sub,k1]
            graph2 = study5_dec_subnetwork_graphs_each_subject[sub,k2]
            similarities_between_DC_and_NF_graphs_of_each_subject_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)

In [ ]:
study1_nf_subnetwork_graphs_each_subject = np.array(study1_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))
study2_nf_subnetwork_graphs_each_subject = np.array(study2_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))
study3_nf_subnetwork_graphs_each_subject = np.array(study3_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))
study4_nf_subnetwork_graphs_each_subject = np.array(study4_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))
study5_nf_subnetwork_graphs_each_subject = np.array(study5_nf_subnetwork_graphs).reshape((num_subjects_ready,num_nf_sessions*num_nf_runs,num_networks, num_networks))

corr_study1 = np.zeros((num_nf_sessions*num_nf_runs, num_nf_sessions*num_nf_runs))
corr_study2 = np.zeros((num_nf_sessions*num_nf_runs, num_nf_sessions*num_nf_runs))
corr_study3 = np.zeros((num_nf_sessions*num_nf_runs, num_nf_sessions*num_nf_runs))
corr_study4 = np.zeros((num_nf_sessions*num_nf_runs, num_nf_sessions*num_nf_runs))
corr_study5 = np.zeros((num_nf_sessions*num_nf_runs, num_nf_sessions*num_nf_runs))

similarities_between_NF_of_different_subjects_dist= []
for k1 in range(0, num_nf_sessions*num_nf_runs):
    for k2 in range(0, num_nf_sessions*num_nf_runs):
        for sub1 in range(0,num_subjects_ready):
            for sub2 in range(0,num_subjects_ready):

                # study 1
                graph1 = study1_nf_subnetwork_graphs_each_subject[sub1,k1]
                graph2 = study1_nf_subnetwork_graphs_each_subject[sub2,k2]

                corr_study1[sub1,sub2] = stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic

            # study 2
                graph1 = study2_nf_subnetwork_graphs_each_subject[sub1,k1]
                graph2 = study2_nf_subnetwork_graphs_each_subject[sub2,k2]
                corr_study2[sub1,sub2] = stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic

            # study 3
                graph1 = study3_nf_subnetwork_graphs_each_subject[sub1,k1]
                graph2 = study3_nf_subnetwork_graphs_each_subject[sub2,k2]
                corr_study3[sub1,sub2] = stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic

            # study 4
                graph1 = study4_nf_subnetwork_graphs_each_subject[sub1,k1]
                graph2 = study4_nf_subnetwork_graphs_each_subject[sub2,k2]
                corr_study4[sub1,sub2] = stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic

            # study 5
                graph1 = study5_nf_subnetwork_graphs_each_subject[sub1,k1]
                graph2 = study5_nf_subnetwork_graphs_each_subject[sub2,k2]
                corr_study5[sub1,sub2] = stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic

        similarities_between_NF_of_different_subjects_dist.append(corr_study1[np.triu_indices(num_subjects_ready, k= 1)])
        similarities_between_NF_of_different_subjects_dist.append(corr_study2[np.triu_indices(num_subjects_ready, k= 1)])
        similarities_between_NF_of_different_subjects_dist.append(corr_study3[np.triu_indices(num_subjects_ready, k= 1)])
        similarities_between_NF_of_different_subjects_dist.append(corr_study4[np.triu_indices(num_subjects_ready, k= 1)])
        similarities_between_NF_of_different_subjects_dist.append(corr_study5[np.triu_indices(num_subjects_ready, k= 1)])

In [ ]:
similarities_between_NF_of_different_studies_dist = []

study1_nf_subnetwork_graphs_each_study = study1_nf_subnetwork_graphs_each_subject.reshape((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks, num_networks))
study2_nf_subnetwork_graphs_each_study = study2_nf_subnetwork_graphs_each_subject.reshape((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks, num_networks))
study3_nf_subnetwork_graphs_each_study = study3_nf_subnetwork_graphs_each_subject.reshape((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks, num_networks))
study4_nf_subnetwork_graphs_each_study = study4_nf_subnetwork_graphs_each_subject.reshape((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks, num_networks))
study5_nf_subnetwork_graphs_each_study = study5_nf_subnetwork_graphs_each_subject.reshape((num_subjects_ready*num_nf_sessions*num_nf_runs,num_networks, num_networks))

# study 1 with others
for k1 in range(0, study1_nf_subnetwork_graphs_each_study.shape[0]):
    for k2 in  range(0, study1_nf_subnetwork_graphs_each_study.shape[0]):
        

        graph1 = study1_nf_subnetwork_graphs_each_study[k1,:,:]
        graph2 = study2_nf_subnetwork_graphs_each_study[k2,:,:]
        similarities_between_NF_of_different_studies_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)
         
        graph1 = study1_nf_subnetwork_graphs_each_study[k1,:,:]
        graph2 = study3_nf_subnetwork_graphs_each_study[k2,:,:]
        similarities_between_NF_of_different_studies_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)

        graph1 = study1_nf_subnetwork_graphs_each_study[k1,:,:]
        graph2 = study4_nf_subnetwork_graphs_each_study[k2,:,:]
        similarities_between_NF_of_different_studies_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)
       
        graph1 = study1_nf_subnetwork_graphs_each_study[k1,:,:]
        graph2 = study5_nf_subnetwork_graphs_each_study[k2,:,:]
        similarities_between_NF_of_different_studies_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)
       

# study 2 with others
for k1 in range(0, study1_nf_subnetwork_graphs_each_study.shape[0]):
    for k2 in  range(0, study1_nf_subnetwork_graphs_each_study.shape[0]):
        

        graph1 = study2_nf_subnetwork_graphs_each_study[k1,:,:]
        graph2 = study3_nf_subnetwork_graphs_each_study[k2,:,:]
        similarities_between_NF_of_different_studies_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)
         
        graph1 = study2_nf_subnetwork_graphs_each_study[k1,:,:]
        graph2 = study4_nf_subnetwork_graphs_each_study[k2,:,:]
        similarities_between_NF_of_different_studies_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)

        graph1 = study2_nf_subnetwork_graphs_each_study[k1,:,:]
        graph2 = study5_nf_subnetwork_graphs_each_study[k2,:,:]
        similarities_between_NF_of_different_studies_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)
       

# study 3 with others
for k1 in range(0, study1_nf_subnetwork_graphs_each_study.shape[0]):
    for k2 in  range(0, study1_nf_subnetwork_graphs_each_study.shape[0]):
        

        graph1 = study3_nf_subnetwork_graphs_each_study[k1,:,:]
        graph2 = study4_nf_subnetwork_graphs_each_study[k2,:,:]
        similarities_between_NF_of_different_studies_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)
         
        graph1 = study3_nf_subnetwork_graphs_each_study[k1,:,:]
        graph2 = study5_nf_subnetwork_graphs_each_study[k2,:,:]
        similarities_between_NF_of_different_studies_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)

# study 4 with others
for k1 in range(0, study1_nf_subnetwork_graphs_each_study.shape[0]):
    for k2 in  range(0, study1_nf_subnetwork_graphs_each_study.shape[0]):
        
        graph1 = study4_nf_subnetwork_graphs_each_study[k1,:,:]
        graph2 = study5_nf_subnetwork_graphs_each_study[k2,:,:]
        similarities_between_NF_of_different_studies_dist.append(stats.pearsonr(graph1.flatten(),graph2.flatten()).statistic)
         

In [ ]:
plt.figure(figsize = (17,8))
ax = sns.boxplot([np.array(similarities_between_NF_graphs_of_each_subject_dist).flatten(), similarities_between_DC_and_NF_graphs_of_each_subject_dist, np.array(similarities_between_NF_of_different_subjects_dist).flatten(), similarities_between_NF_of_different_studies_dist], palette = ['green','green','green','green'], width = 0.2)
ax.set_xticklabels( ['NF\n same subject', 'NF and Baseline \n same subject', 'NF \n different subjects','NF \n different studies'], fontsize=x_ticks_font)
plt.yticks([-0.4,0,0.4,0.8],fontsize = y_ticks_font)
plt.grid(True)
plt.ylabel('Causal Graph Similarity', fontsize = y_label_font)

x1, x2 = 0.25, 0.7
y, h, col =0.85, 0.03, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "***", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 1.25, 1.7
y, h, col =0.7, 0.03, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "***", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 2.25, 2.7
y, h, col =0.6, 0.03, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "***", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

plt.savefig(os.path.join(FIG_MAIN, 'Fig1c_similarity_hierarchy.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig1c_similarity_hierarchy.png'),bbox_inches='tight')

print(stats.ranksums(np.array(similarities_between_NF_graphs_of_each_subject_dist).flatten(), similarities_between_DC_and_NF_graphs_of_each_subject_dist, alternative= 'greater'))
print(stats.ranksums(similarities_between_DC_and_NF_graphs_of_each_subject_dist, np.array(similarities_between_NF_of_different_subjects_dist).flatten(), alternative= 'greater'))
print(stats.ranksums(np.array(similarities_between_NF_of_different_subjects_dist).flatten(),similarities_between_NF_of_different_studies_dist, alternative= 'greater'))

# Cognitive vs perceptive neurofeedback (Figure 4, Supplementary Fig. 10)

In [ ]:
higher_level_cognition_nf_minus_dc_partial_corr = []
lower_level_cognition_nf_minus_dc_partial_corr = []

for sub in range(0,num_subjects_ready):
    for ses in range(0, num_nf_sessions):
        for r in range(0,num_nf_runs):
            nf_graph = np.array(study1_nf_partial_corr_subnetwork_graphs)[sub, ses,r,:,:]
            dec_graph= np.mean(np.array(study1_decoder_partial_corr_subnetwork_graphs)[sub,0,:,:,:], axis = 0)
            higher_level_cognition_nf_minus_dc_partial_corr.append(nf_graph-dec_graph)

            nf_graph = np.array(study4_nf_partial_corr_subnetwork_graphs)[sub, ses,r,:,:]
            dec_graph= np.mean(np.array(study4_decoder_partial_corr_subnetwork_graphs)[sub,0,:,:,:], axis = 0)
            higher_level_cognition_nf_minus_dc_partial_corr.append(nf_graph-dec_graph)

            nf_graph = np.array(study5_nf_partial_corr_subnetwork_graphs)[sub, ses,r,:,:]
            dec_graph= np.mean(np.array(study5_decoder_partial_corr_subnetwork_graphs)[sub,0,:,:,:], axis = 0)
            higher_level_cognition_nf_minus_dc_partial_corr.append(nf_graph-dec_graph)

            nf_graph = np.array(study2_nf_partial_corr_subnetwork_graphs)[sub, ses,r,:,:]
            dec_graph= np.mean(np.array(study2_decoder_partial_corr_subnetwork_graphs)[sub,0,:,:,:], axis = 0)
            lower_level_cognition_nf_minus_dc_partial_corr.append(nf_graph-dec_graph)

            nf_graph = np.array(study3_nf_partial_corr_subnetwork_graphs)[sub, ses,r,:,:]
            dec_graph= np.mean(np.array(study3_decoder_partial_corr_subnetwork_graphs)[sub,0,:,:,:], axis = 0)
            lower_level_cognition_nf_minus_dc_partial_corr.append(nf_graph-dec_graph)

In [ ]:
higher_lower_level_compare_edge_strength_statistics = np.zeros((num_networks, num_networks))
higher_lower_level_compare_edge_strength_pvalue = np.zeros((num_networks, num_networks))

for i in range(0,num_networks):
    for j in range(0, num_networks):

        x = np.array(higher_level_cognition_nf_minus_dc_partial_corr)[:,i,j]
        y = np.array(lower_level_cognition_nf_minus_dc_partial_corr)[:,i,j]

        temp = scipy.stats.ranksums(x, y, alternative='two-sided')
        higher_lower_level_compare_edge_strength_statistics[i,j] = temp.statistic
        higher_lower_level_compare_edge_strength_pvalue[i,j]= temp.pvalue

higher_lower_level_compare_edge_pvalue_corrected = np.reshape(statsmodels.stats.multitest.fdrcorrection(higher_lower_level_compare_edge_strength_pvalue.flatten())[0], (num_networks, num_networks))

In [ ]:
plt.figure(figsize = (7,7))
ax1 = plt.subplot()
im = plt.imshow(np.multiply(higher_lower_level_compare_edge_strength_statistics, higher_lower_level_compare_edge_pvalue_corrected), cmap = 'coolwarm', vmin = -np.max(np.abs(higher_lower_level_compare_edge_strength_statistics)),  vmax = np.max(np.abs(higher_lower_level_compare_edge_strength_statistics)))

ax1.set_xticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 14)
ax1.set_yticks(np.arange(len(visualization_labels)), labels = visualization_labels , fontsize = 14)
g = plt.setp(ax1.get_xticklabels(), rotation =45, ha = "right",
        rotation_mode="anchor")

divider = make_axes_locatable(ax1)
cax = divider.append_axes("right", size="5%", pad=0.3)
clb = plt.colorbar(im, cax=cax)
clb.ax.tick_params(labelsize=12) 

plt.savefig(os.path.join(FIG_MAIN, 'Fig4a_cognitive_vs_perceptive.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig4a_cognitive_vs_perceptive.png'),bbox_inches='tight')

In [ ]:
temp = np.multiply(higher_lower_level_compare_edge_strength_statistics, higher_lower_level_compare_edge_pvalue_corrected)

# connections to somatomotor
higher_lower_sommot = np.zeros((num_networks, num_networks))
higher_lower_sommot[14:16,:] = temp[14:16,:]
higher_lower_sommot[:,14:16] = temp[:,14:16]

# connections to dorsal attention
higher_lower_dors_att = np.zeros((num_networks, num_networks))
higher_lower_dors_att[12:14,:] = temp[12:14,:]
higher_lower_dors_att[:,12:14] = temp[:,12:14]

# connections to subcortical
higher_lower_sub = np.zeros((num_networks, num_networks))
higher_lower_sub[0:2,:] = temp[0:2,:]
higher_lower_sub[:,0:2] = temp[:,0:2]

# other connections
higher_lower_others = np.zeros((num_networks, num_networks))
higher_lower_others[2:12,2:12] = temp[2:12,2:12]
higher_lower_others[2:12,2:12] = temp[2:12,2:12]

G = nx.from_numpy_array(higher_lower_sommot,create_using=nx.DiGraph)
edges_of_whole_matrix,weights_of_whole_matrix= zip(*nx.get_edge_attributes(G, 'weight').items())
norm = plt.Normalize(vmin=np.min(weights_of_whole_matrix), vmax=np.max(weights_of_whole_matrix))

# Create a NetworkX graph from the adjacency matrix
G_higher_lower_sommot= nx.from_numpy_array(higher_lower_sommot,create_using=nx.DiGraph)
G_higher_lower_dors_att = nx.from_numpy_array(higher_lower_dors_att,create_using=nx.DiGraph)
G_higher_lower_sub= nx.from_numpy_array(higher_lower_sub,create_using=nx.DiGraph)
G_higher_lower_others= nx.from_numpy_array(higher_lower_others,create_using=nx.DiGraph)

visualization_labels
node_locs = np.array([[-15, -40, 10],[15, -40, 10], #sub
                      [-22, -90, 10],[22, -90, 10],  # Vis
                      [-35, -60, 10],[35, -60, 10], # Lim
                      [-10, 50, 10],[10, 50, 10],  # Cont
                      [-30, 30, 10],[30, 30, 10],  # DMN
                      [-40, -0, 10],[40, -0, 10], # Vent
                      [-10, -0, 10],[10, -0, 10], # Dors
                      [-35, -20, 10],[35, -20, 10], # Som-Mot
                      ])

node_positions_2d = node_locs[:, :2]
node_positions = {i: (x, y) for i, (x, y) in enumerate(node_positions_2d)}
display_mode = 'z'
node_size = 4000

In [ ]:
# higher_lower_sommot

edges,weights= zip(*nx.get_edge_attributes(G_higher_lower_sommot, 'weight').items())
edge_colors = plt.cm.coolwarm(norm(weights))

figure = plt.figure(figsize=(10, 10))
display = plotting.plot_glass_brain(None, display_mode=display_mode,
                                    figure=figure, annotate=False, alpha=0.15)

nx.draw_networkx_edges(G_higher_lower_sommot, pos=node_positions, edge_color=edge_colors, width=3, node_size=node_size, arrows=True, arrowsize=12)
# # Draw nodes
nx.draw_networkx_nodes(G_higher_lower_sommot, pos=node_positions, node_color='lightgray', node_size=node_size)
node_labels_dict = {i: label for i, label in enumerate(short_label_modified)}
nx.draw_networkx_labels(G_higher_lower_sommot, pos=node_positions, labels=node_labels_dict, font_size=18, font_color='black', font_weight='bold')

plt.text(-15, 67, 'L', fontsize=26, ha='center', va='center', fontweight = 'bold')
plt.text(20, 67, 'R', fontsize=26, ha='center', va='center', fontweight = 'bold')

plt.savefig(os.path.join(FIG_MAIN, 'Fig4b_somatomotor.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig4b_somatomotor.png'),bbox_inches='tight')

# dors-att

edges,weights= zip(*nx.get_edge_attributes(G_higher_lower_dors_att, 'weight').items())
edge_colors = plt.cm.coolwarm(norm(weights))

figure = plt.figure(figsize=(10, 10))
display = plotting.plot_glass_brain(None, display_mode=display_mode,
                                    figure=figure, annotate=False, alpha=0.15)

nx.draw_networkx_edges(G_higher_lower_dors_att, pos=node_positions, edge_color=edge_colors, width=3, node_size=node_size, arrows=True, arrowsize=12)
# # Draw nodes
nx.draw_networkx_nodes(G_higher_lower_dors_att, pos=node_positions, node_color='lightgray', node_size=node_size)
node_labels_dict = {i: label for i, label in enumerate(short_label_modified)}
nx.draw_networkx_labels(G_higher_lower_dors_att, pos=node_positions, labels=node_labels_dict, font_size=18, font_color='black', font_weight='bold')

plt.text(-15, 67, 'L', fontsize=26, ha='center', va='center', fontweight = 'bold')
plt.text(20, 67, 'R', fontsize=26, ha='center', va='center', fontweight = 'bold')

plt.savefig(os.path.join(FIG_MAIN, 'Fig4c_dorsal_attention.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig4c_dorsal_attention.png'),bbox_inches='tight')

# subcortical

edges,weights= zip(*nx.get_edge_attributes(G_higher_lower_sub, 'weight').items())
edge_colors = plt.cm.coolwarm(norm(weights))

figure = plt.figure(figsize=(10, 10))
display = plotting.plot_glass_brain(None, display_mode=display_mode,
                                    figure=figure, annotate=False, alpha=0.15)

nx.draw_networkx_edges(G_higher_lower_sub, pos=node_positions, edge_color=edge_colors, width=3, node_size=node_size, arrows=True, arrowsize=12)
# # Draw nodes
nx.draw_networkx_nodes(G_higher_lower_sub, pos=node_positions, node_color='lightgray', node_size=node_size)
node_labels_dict = {i: label for i, label in enumerate(short_label_modified)}
nx.draw_networkx_labels(G_higher_lower_sub, pos=node_positions, labels=node_labels_dict, font_size=18, font_color='black', font_weight='bold')

plt.text(-15, 67, 'L', fontsize=26, ha='center', va='center', fontweight = 'bold')
plt.text(20, 67, 'R', fontsize=26, ha='center', va='center', fontweight = 'bold')

plt.savefig(os.path.join(FIG_MAIN, 'Fig4d_subcortical.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig4d_subcortical.png'),bbox_inches='tight')

# others

edges,weights= zip(*nx.get_edge_attributes(G_higher_lower_others, 'weight').items())
edge_colors = plt.cm.coolwarm(norm(weights))

figure = plt.figure(figsize=(10, 10))
display = plotting.plot_glass_brain(None, display_mode=display_mode,
                                    figure=figure, annotate=False, alpha=0.15)

nx.draw_networkx_edges(G_higher_lower_others, pos=node_positions, edge_color=edge_colors, width=3, node_size=node_size, arrows=True, arrowsize=12)
# # Draw nodes
nx.draw_networkx_nodes(G_higher_lower_others, pos=node_positions, node_color='lightgray', node_size=node_size)
node_labels_dict = {i: label for i, label in enumerate(short_label_modified)}
nx.draw_networkx_labels(G_higher_lower_others, pos=node_positions, labels=node_labels_dict, font_size=18, font_color='black', font_weight='bold')

plt.text(-15, 67, 'L', fontsize=26, ha='center', va='center', fontweight = 'bold')
plt.text(20, 67, 'R', fontsize=26, ha='center', va='center', fontweight = 'bold')

In [ ]:
higher_level_cognition_nf_partial_corr_subnetwork_graphs = np.concatenate((np.array(study1_nf_partial_corr_subnetwork_graphs).reshape(-1,16,16), np.array(study4_nf_partial_corr_subnetwork_graphs).reshape(-1,16,16), np.array(study5_nf_partial_corr_subnetwork_graphs).reshape(-1,16,16)), axis = 0  ) 
lower_level_cognition_nf_partial_corr_subnetwork_graphs = np.concatenate(( np.array(study2_nf_partial_corr_subnetwork_graphs).reshape(-1,16,16), np.array(study3_nf_partial_corr_subnetwork_graphs).reshape(-1,16,16)), axis = 0  ) 

higher_level_cognition_nf_subnetwork_graphs = np.concatenate((np.array(study1_nf_subnetwork_graphs).reshape(-1,16,16), np.array(study4_nf_subnetwork_graphs).reshape(-1,16,16), np.array(study5_nf_subnetwork_graphs).reshape(-1,16,16)), axis = 0  ) 
lower_level_cognition_nf_subnetwork_graphs = np.concatenate(( np.array(study2_nf_subnetwork_graphs).reshape(-1,16,16), np.array(study3_nf_subnetwork_graphs).reshape(-1,16,16)), axis = 0  ) 

### Degree

In [ ]:
higher_level_studies_degree = np.vstack((study1_nf_subnetwork_graphs_degree,study4_nf_subnetwork_graphs_degree,study5_nf_subnetwork_graphs_degree))
lower_level_studies_degree = np.vstack((study2_nf_subnetwork_graphs_degree,study3_nf_subnetwork_graphs_degree))

higher_level_studies_causal_flow = np.vstack((study1_nf_subnetwork_graphs_causal_flow,study4_nf_subnetwork_graphs_causal_flow,study5_nf_subnetwork_graphs_causal_flow))
lower_level_studies_causal_flow = np.vstack((study2_nf_subnetwork_graphs_causal_flow,study3_nf_subnetwork_graphs_causal_flow))

In [ ]:
data1_long = pd.DataFrame(lower_level_studies_degree, columns=visualization_labels).melt( value_name='Value')
data1_long['Type'] = 'Perception'
data2_long = pd.DataFrame(higher_level_studies_degree , columns=visualization_labels).melt(value_name='Value')
data2_long['Type'] = 'Cognition'

combined_data = pd.concat([data1_long, data2_long])

# Plot
plt.figure(figsize=(12, 6))
ax = sns.barplot(x='variable', y='Value', hue='Type', data=combined_data, palette=['lightblue', 'blue'], errorbar= 'se')

# Customize labels and title
plt.ylabel('Nodal Degree', fontsize = 22)
plt.grid(True)
plt.xticks(rotation=45, fontsize = 18)
plt.yticks(fontsize = 18)
plt.xlabel('')

# sub
x1, x2 = -0.3, 0.3
y, h, col = 0.41, 0.003, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 0.7, 1.3
y, h, col = 0.41, 0.003, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

#vis
x1, x2 = 1.7, 2.3
y, h, col = 0.51, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "***", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 2.7, 3.3
y, h, col =0.51, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "*", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

# lim
x1, x2 = 3.7, 4.3
y, h, col =0.46, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 4.7, 5.3
y, h, col =0.46, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

# cont
x1, x2 = 5.7, 6.3
y, h, col = 0.55, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 6.7, 7.3
y, h, col =0.55, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

# dmn
x1, x2 = 7.7, 8.3
y, h, col = 0.55, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 8.7, 9.3
y, h, col =0.55, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

# vent-att
x1, x2 = 9.7, 10.3
y, h, col = 0.63, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 10.7, 11.3
y, h, col =0.65, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

# dors-att
x1, x2 = 11.7, 12.3
y, h, col = 0.62, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 12.7, 13.3
y, h, col =0.62, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

# som-mot
x1, x2 = 13.7, 14.3
y, h, col = 0.62, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 14.7, 15.3
y, h, col =0.62, 0.005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x_left, x_right = plt.xlim()
plt.xlim(-0.5, x_right-0.5)

plt.legend( loc='upper left', fontsize = 18 )

plt.tight_layout()
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig10b_cognitive_vs_perceptive_degree.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig10b_cognitive_vs_perceptive_degree.png'),bbox_inches='tight')

In [ ]:
data1_long = pd.DataFrame(lower_level_studies_causal_flow, columns=visualization_labels).melt( value_name='Value')
data1_long['Type'] = 'Perception'
data2_long = pd.DataFrame(higher_level_studies_causal_flow, columns=visualization_labels).melt(value_name='Value')
data2_long['Type'] = 'Cognition'

combined_data = pd.concat([data1_long, data2_long])

# Plot
plt.figure(figsize=(12, 6))
ax = sns.barplot(x='variable', y='Value', hue='Type', data=combined_data, palette=['lightblue', 'blue'],  errorbar= 'se')

# Customize labels and title
plt.ylabel('Net Flow', fontsize = 22)
plt.grid(True)
plt.xticks(rotation=45, fontsize = 16)
plt.yticks(fontsize = 16)
plt.xlabel('')

# sub
x1, x2 = -0.3, 0.3
y, h, col = -0.0018, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-3*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 0.7, 1.3
y, h, col = -0.0018, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-3*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

# vis
x1, x2 = 1.7, 2.3
y, h, col = -0.0018, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-3*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 2.7, 3.3
y, h, col =-0.0018, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-3*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

#lim
x1, x2 = 3.7, 4.3
y, h, col = 0.002, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 4.7, 5.3
y, h, col =-0.0015, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-4*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

#cont
x1, x2 = 5.7, 6.3
y, h, col = 0.001, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 6.7, 7.3
y, h, col =-0.0012, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-3*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

# dmn
x1, x2 = 7.7, 8.3
y, h, col = 0.0007, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 8.7, 9.3
y, h, col =-0.0012, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-3*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

#vent-att
x1, x2 = 9.7, 10.3
y, h, col = 0.002, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 10.7, 11.3
y, h, col =-0.0015, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-4*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

# dors-att
x1, x2 = 11.7, 12.3
y, h, col = 0.0015, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 12.7, 13.3
y, h, col = 0.002, 0.0001, 'k'

plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 12.7, 13.3

# som-mot
x1, x2 = 13.7, 14.3
y, h, col = 0.0015, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 14.7, 15.3
y, h, col =-0.0015, 0.0001, 'k'
plt.plot([x1, x1, x2, x2], [y, y-h, y-h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y-4*h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x_left, x_right = plt.xlim()
plt.xlim(-0.5, x_right-0.5)

plt.legend( loc='upper left', fontsize = 16)
 
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig10c_cognitive_vs_perceptive_net_flow.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig10c_cognitive_vs_perceptive_net_flow.png'),bbox_inches='tight')

## Global network measures

In [ ]:
# Higher level
higher_level_nf_subnetwork_graphs_density = []
higher_level_nf_subnetwork_graphs_shortest_path_length =  []
higher_level_nf_subnetwork_graphs_assortativity = []
higher_level_nf_subnetwork_graphs_modularity = []
higher_level_nf_subnetwork_graphs_spectral_radius = []
higher_level_nf_subnetwork_graphs_synchronizability = []

for s in range(0,np.array(higher_level_cognition_nf_subnetwork_graphs).shape[0]):

    weighted_matrix = np.array(higher_level_cognition_nf_subnetwork_graphs)[s,:,:]

    G = funcs.create_weighted_digraph_from_matrix(weighted_matrix)

    higher_level_nf_subnetwork_graphs_density.append(np.mean(weighted_matrix))
    
    higher_level_nf_subnetwork_graphs_shortest_path_length.append(nx.average_shortest_path_length(G))
    
    higher_level_nf_subnetwork_graphs_assortativity.append(nx.degree_assortativity_coefficient(G, weight='weight'))
    
    G_undirected = G.to_undirected()
    partition = nx.community.greedy_modularity_communities(G_undirected, weight='weight')
    modularity = nx.community.modularity(G_undirected, partition, weight='weight')
    
    higher_level_nf_subnetwork_graphs_modularity.append(modularity)
    
    eigenvalues = np.linalg.eigvals(weighted_matrix)
    max_abs_eigenvalue = max(abs(eigenvalues))
    higher_level_nf_subnetwork_graphs_spectral_radius.append(max_abs_eigenvalue)

    L = nx.linalg.laplacianmatrix.directed_laplacian_matrix(G, weight='weight')
    eigenvalues, _ = np.linalg.eig(L)
    eigenvalues_sorted = np.sort(eigenvalues)
    lambda_L2 = eigenvalues_sorted[1]  # Second smallest eigenvalue
    lambda_Lmax = eigenvalues_sorted[-1]  # Largest eigenvalue
    S = lambda_L2 / lambda_Lmax
    higher_level_nf_subnetwork_graphs_synchronizability.append(S)

# Lower level

lower_level_nf_subnetwork_graphs_density = []
lower_level_nf_subnetwork_graphs_shortest_path_length =  []
lower_level_nf_subnetwork_graphs_assortativity = []
lower_level_nf_subnetwork_graphs_modularity = []
lower_level_nf_subnetwork_graphs_spectral_radius = []
lower_level_nf_subnetwork_graphs_synchronizability = []

for s in range(0,np.array(lower_level_cognition_nf_subnetwork_graphs).shape[0]):
    weighted_matrix = np.array(lower_level_cognition_nf_subnetwork_graphs)[s,:,:]

    lower_level_nf_subnetwork_graphs_density.append(np.mean(weighted_matrix))
    
    lower_level_nf_subnetwork_graphs_shortest_path_length.append(nx.average_shortest_path_length(G))
    
    lower_level_nf_subnetwork_graphs_assortativity.append(nx.degree_assortativity_coefficient(G, weight='weight'))
    
    G_undirected = G.to_undirected()
    partition = nx.community.greedy_modularity_communities(G_undirected, weight='weight')
    modularity = nx.community.modularity(G_undirected, partition, weight='weight')
    
    lower_level_nf_subnetwork_graphs_modularity.append(modularity)
    
    eigenvalues = np.linalg.eigvals(weighted_matrix)
    max_abs_eigenvalue = max(abs(eigenvalues))
    lower_level_nf_subnetwork_graphs_spectral_radius.append(max_abs_eigenvalue)

    L = nx.linalg.laplacianmatrix.directed_laplacian_matrix(G, weight='weight')
    eigenvalues, _ = np.linalg.eig(L)
    eigenvalues_sorted = np.sort(eigenvalues)
    lambda_L2 = eigenvalues_sorted[1]  # Second smallest eigenvalue
    lambda_Lmax = eigenvalues_sorted[-1]  # Largest eigenvalue
    S = lambda_L2 / lambda_Lmax
    lower_level_nf_subnetwork_graphs_synchronizability.append(S)

In [ ]:
higher_all_global_measures_nf = np.vstack((higher_level_nf_subnetwork_graphs_density,higher_level_nf_subnetwork_graphs_shortest_path_length,higher_level_nf_subnetwork_graphs_assortativity,higher_level_nf_subnetwork_graphs_modularity,higher_level_nf_subnetwork_graphs_spectral_radius,higher_level_nf_subnetwork_graphs_synchronizability)).T
print(higher_all_global_measures_nf.shape)

lower_all_global_measures_nf = np.vstack((lower_level_nf_subnetwork_graphs_density,lower_level_nf_subnetwork_graphs_shortest_path_length,lower_level_nf_subnetwork_graphs_assortativity,lower_level_nf_subnetwork_graphs_modularity,lower_level_nf_subnetwork_graphs_spectral_radius,lower_level_nf_subnetwork_graphs_synchronizability)).T
print(lower_all_global_measures_nf.shape)

data1_long = pd.DataFrame(lower_all_global_measures_nf, columns=global_measures_labels).melt( value_name='Value')
data1_long['Type'] = 'Perception'
data2_long = pd.DataFrame(higher_all_global_measures_nf, columns=global_measures_labels).melt(value_name='Value')
data2_long['Type'] = 'Cognition'

combined_data = pd.concat([data1_long, data2_long])

# Plot
plt.figure(figsize=(12, 6))
ax = sns.barplot(x='variable', y='Value', hue='Type', data=combined_data, palette=['lightblue', 'blue'], errorbar= 'se',width = 0.4)

# Customize labels and title
plt.grid(True)
plt.xticks(rotation=0, fontsize = 18)
plt.yticks(fontsize = 18)

plt.xlabel('')
plt.ylabel('')
plt.yscale('log')

x1, x2 = -0.1, 0.1
y, h, col =0.3, 0.015, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 0.9, 1.1
y, h, col =1.3, 0.03, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "***", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 1.9, 2.1
y, h, col =0.03, 0.001, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "***", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 2.9, 3.1
y, h, col =0.4, 0.015, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "***", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 3.9, 4.1
y, h, col =5, 0.3, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 4.9, 5.1
y, h, col =0.6, 0.03, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "***", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x_left, x_right = plt.xlim()
plt.xlim(-0.3, x_right-0.2)

plt.legend( loc='upper right', fontsize = 15, title_fontsize = 20)

plt.tight_layout()
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig10a_cognitive_vs_perceptive_global_measures.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig10a_cognitive_vs_perceptive_global_measures.png'),bbox_inches='tight')

# Correlations with neurofeedback score (Figure 3, Supplementary Figs. 7-8)

## Edge Level

In [ ]:
correlations_with_scores = np.zeros((num_networks, num_networks))
correlations_with_scores_pvalue = np.zeros((num_networks, num_networks))

for i in range(0,num_networks):
    for j in range(0,num_networks):

        t = scipy.stats.spearmanr(np.array(all_studies_nf_partial_corr_subnetworks)[:,i,j],all_studies_nf_score_mean, alternative='two-sided', nan_policy='omit')
        correlations_with_scores[i,j] = t.statistic
        correlations_with_scores_pvalue[i,j] = t.pvalue
 
correlations_with_scores_significance = np.reshape(statsmodels.stats.multitest.fdrcorrection(correlations_with_scores_pvalue.flatten())[0], (num_networks, num_networks))

# Global network measures vs score (Supplementary Fig. 8a)

### Density

In [ ]:
all_studies_nf_density = []
for s in range(0,np.array(all_studies_nf_subnetworks).shape[0]):

    graph = np.array(all_studies_nf_subnetworks)[s,:,:]
    all_studies_nf_density.append(np.mean(graph))

In [ ]:
plt.figure(figsize = (15,5))
plt.subplot(1,2,1)

density_and_score_change_corr= scipy.stats.spearmanr(np.array(all_studies_nf_density),all_studies_nf_score_changes, alternative='two-sided', nan_policy='omit')
print(density_and_score_change_corr)
plt.scatter(np.array(all_studies_nf_density),all_studies_nf_score_changes )
plt.xlabel('Graph Density')
plt.ylabel('Changes in NF Score')
plt.grid()

plt.subplot(1,2,2)
density_and_score_mean_corr= scipy.stats.spearmanr(np.array(all_studies_nf_density),all_studies_nf_score_mean, alternative='two-sided', nan_policy='omit')
print(density_and_score_mean_corr)
plt.scatter(np.array(all_studies_nf_density),all_studies_nf_score_mean)
plt.xlabel('Graph Density')
plt.ylabel('Average NF Score')
plt.grid()

### Average shortest path length

In [ ]:
average_shortest_path_length_all = []
for s in range(0,np.array(all_studies_nf_subnetworks).shape[0]):
    weighted_matrix = np.array(all_studies_nf_subnetworks)[s,:,:]
    G = funcs.create_weighted_digraph_from_matrix(weighted_matrix)
    average_shortest_path_length_all.append(nx.average_shortest_path_length(G))

In [ ]:
shortest_path_length_and_score_mean_corr= scipy.stats.spearmanr(np.array(average_shortest_path_length_all),all_studies_nf_score_mean, alternative='two-sided', nan_policy='omit')

print(shortest_path_length_and_score_mean_corr)
plt.scatter(np.array(average_shortest_path_length_all),all_studies_nf_score_changes )
plt.xlabel('Average Shortest Path Length')
plt.ylabel('Average NF Score')
plt.grid()

### Assortativity

In [ ]:
assortativity_all = []
for s in range(0,np.array(all_studies_nf_subnetworks).shape[0]):
    weighted_matrix = np.array(all_studies_nf_subnetworks)[s,:,:]
    G = funcs.create_weighted_digraph_from_matrix(weighted_matrix)
    assortativity_all.append(nx.degree_assortativity_coefficient(G, weight='weight'))

In [ ]:
plt.figure(figsize = (15,5))
plt.subplot(1,2,1)

assortativity_and_score_change_corr= scipy.stats.spearmanr(np.array(assortativity_all),all_studies_nf_score_changes, alternative='two-sided', nan_policy='omit')
print(assortativity_and_score_change_corr)
plt.scatter(np.array(assortativity_all),all_studies_nf_score_changes )
plt.xlabel('Graph Assortativity')
plt.ylabel('Changes in NF Score')
plt.grid()

plt.subplot(1,2,2)
assortativity_and_score_mean_corr= scipy.stats.spearmanr(np.array(assortativity_all),all_studies_nf_score_mean, alternative='two-sided', nan_policy='omit')
print(assortativity_and_score_mean_corr)
plt.scatter(np.array(assortativity_all),all_studies_nf_score_mean)
plt.xlabel('Graph Assortativity')
plt.ylabel('Average NF Score')
plt.grid()

### Modularity

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities
modularity_all = []
for s in range(0,np.array(all_studies_nf_subnetworks).shape[0]):
    weighted_matrix = np.array(all_studies_nf_subnetworks)[s,:,:]
    G = funcs.create_weighted_digraph_from_matrix(weighted_matrix)
    G_undirected = G.to_undirected()
    partition = nx.community.greedy_modularity_communities(G_undirected, weight='weight')
    modularity = nx.community.modularity(G_undirected, partition, weight='weight')
    modularity_all.append(modularity)

In [ ]:
plt.figure(figsize = (15,5))
plt.subplot(1,2,1)

modularity_and_score_change_corr= scipy.stats.spearmanr(np.array(modularity_all),all_studies_nf_score_changes, alternative='two-sided', nan_policy='omit')
print(modularity_and_score_change_corr)
plt.scatter(np.array(modularity_all),all_studies_nf_score_changes )
plt.xlabel('Graph modularity')
plt.ylabel('Changes in NF Score')
plt.grid()

plt.subplot(1,2,2)
modularity_and_score_mean_corr= scipy.stats.spearmanr(np.array(modularity_all),all_studies_nf_score_mean, alternative='two-sided', nan_policy='omit')
print(modularity_and_score_mean_corr)
plt.scatter(np.array(modularity_all),all_studies_nf_score_mean)
plt.xlabel('Graph modularity')
plt.ylabel('Average NF Score')
plt.grid()

### Spectral Radius

In [ ]:
spectral_radius_all = []
for s in range(0,np.array(all_studies_nf_subnetworks).shape[0]):
    weighted_matrix = np.array(all_studies_nf_subnetworks)[s,:,:]
    G = funcs.create_weighted_digraph_from_matrix(weighted_matrix)
    eigenvalues = np.linalg.eigvals(weighted_matrix)
    max_abs_eigenvalue = max(abs(eigenvalues))
    spectral_radius_all.append(max_abs_eigenvalue)

In [ ]:
plt.figure(figsize = (15,5))
plt.subplot(1,2,1)

spectral_radius_and_score_change_corr= scipy.stats.spearmanr(np.array(spectral_radius_all),all_studies_nf_score_changes, alternative='two-sided', nan_policy='omit')
print(spectral_radius_and_score_change_corr)
plt.scatter(np.array(spectral_radius_all),all_studies_nf_score_changes )
plt.xlabel('Graph Spectral Radius')
plt.ylabel('Changes in NF Score')
plt.grid()

plt.subplot(1,2,2)
spectral_radius_and_score_mean_corr= scipy.stats.spearmanr(np.array(spectral_radius_all),all_studies_nf_score_mean, alternative='two-sided', nan_policy='omit')
print(spectral_radius_and_score_mean_corr)
plt.scatter(np.array(spectral_radius_all),all_studies_nf_score_mean)
plt.xlabel('Graph Spectral Radius')
plt.ylabel('Average NF Score')
plt.grid()

### Synchronizability

In [ ]:
synchronizability_all = []
for s in range(0,np.array(all_studies_nf_subnetworks).shape[0]):
    weighted_matrix = np.array(all_studies_nf_subnetworks)[s,:,:]
    G = funcs.create_weighted_digraph_from_matrix(weighted_matrix)
    L = nx.linalg.laplacianmatrix.directed_laplacian_matrix(G, weight='weight')
    eigenvalues, _ = np.linalg.eig(L)
    eigenvalues_sorted = np.sort(eigenvalues)
    lambda_L2 = eigenvalues_sorted[1]  # Second smallest eigenvalue
    lambda_Lmax = eigenvalues_sorted[-1]  # Largest eigenvalue

    S = lambda_L2 / lambda_Lmax
    synchronizability_all.append(S)

In [ ]:
plt.figure(figsize = (15,5))
plt.subplot(1,2,1)

synchronizability_and_score_change_corr= scipy.stats.spearmanr(np.array(synchronizability_all),all_studies_nf_score_changes, alternative='two-sided', nan_policy='omit')
print(synchronizability_and_score_change_corr)
plt.scatter(np.array(synchronizability_all),all_studies_nf_score_changes )
plt.xlabel('Graph Synchronizability')
plt.ylabel('Changes in NF Score')
plt.grid()

plt.subplot(1,2,2)
synchronizability_and_score_mean_corr= scipy.stats.spearmanr(np.array(synchronizability_all),all_studies_nf_score_mean, alternative='two-sided', nan_policy='omit')
print(synchronizability_and_score_mean_corr)
plt.scatter(np.array(synchronizability_all),all_studies_nf_score_mean)
plt.xlabel('Graph Synchronizability')
plt.ylabel('Average NF Score')
plt.grid()

In [ ]:
global_measures_corr_with_mean_score = [density_and_score_mean_corr.statistic,shortest_path_length_and_score_mean_corr.statistic,assortativity_and_score_mean_corr.statistic,modularity_and_score_mean_corr.statistic,spectral_radius_and_score_mean_corr.statistic,synchronizability_and_score_mean_corr.statistic]
print(global_measures_corr_with_mean_score)

data = pd.DataFrame({
    'Label': global_measures_labels,
    'Value': global_measures_corr_with_mean_score
})

# Plot the data
plt.figure(figsize=(12, 6))
ax = sns.barplot(x='Label', y='Value', data=data, color='orange', width = 0.2)
# Customize labels and title
plt.grid(True)
plt.xticks(rotation=30, fontsize = 18)
plt.yticks(fontsize = 18)

plt.xlabel('')
plt.ylabel('Correlation with Score', fontsize = 22)

x1, x2 = -0.1, 0.1
y, h, col =-0.045, 0.00, 'k'
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 0.9, 1.1
y, h, col =0.055, 0.00, 'k'
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 1.9, 2.1
y, h, col =-0.05, 0.00, 'k'
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 2.9, 3.1
y, h, col =0.058, 0.0, 'k'
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 3.9, 4.1
y, h, col =-0.043, 0.00, 'k'
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x1, x2 = 4.9, 5.1
y, h, col =-0.032, 0.0, 'k'
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

x_left, x_right = plt.xlim()
plt.xlim(-0.2, x_right-0.3)

    
plt.tight_layout()
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig08a_global_measures_vs_score.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig08a_global_measures_vs_score.png'),bbox_inches='tight')

print(shortest_path_length_and_score_mean_corr)

## Nodal measures vs score (Supplementary Fig. 8b,c)

### Degree Centrality

In [ ]:
degree_and_score_change_corr_statistic = []
degree_and_score_change_corr_pvalue = []

degree_and_score_mean_corr_statistic = []
degree_and_score_mean_corr_pvalue = []

for n in range(0,num_networks):

    degree =  np.mean(np.array(all_studies_nf_subnetworks)[:,n,:], axis = 1)+ np.mean(np.array(all_studies_nf_subnetworks)[:,:,n], axis = 1)
    
    temp1 = scipy.stats.spearmanr(degree,all_studies_nf_score_changes, alternative='two-sided', nan_policy='omit')
    temp1_statistic = temp1.statistic
    temp1_pvalue = temp1.pvalue
    
    degree_and_score_change_corr_statistic.append(temp1_statistic)
    degree_and_score_change_corr_pvalue.append(temp1_pvalue)

    temp2 = scipy.stats.spearmanr(degree,all_studies_nf_score_mean, alternative='two-sided', nan_policy='omit')
    temp2_statistic = temp2.statistic
    temp2_pvalue = temp2.pvalue

    degree_and_score_mean_corr_statistic.append(temp2_statistic)
    degree_and_score_mean_corr_pvalue.append(temp2_pvalue)

indegree_and_score_change_corr_pvalue_corrected = statsmodels.stats.multitest.fdrcorrection(degree_and_score_change_corr_pvalue)[0]
indegree_and_score_mean_corr_pvalue_corrected = statsmodels.stats.multitest.fdrcorrection(degree_and_score_mean_corr_pvalue)[0]

plt.figure(figsize = (15,5))
plt.subplot(1,2,1)
plt.stem(np.multiply(degree_and_score_change_corr_statistic,indegree_and_score_change_corr_pvalue_corrected) )
plt.xticks(np.arange(0,num_networks), labels = visualization_labels, rotation = 70)
plt.ylabel('Significant Correlations with Changes in NF Score')
plt.grid()
plt.title('Degree Centrality')
plt.ylim([-0.15,0.15])

plt.subplot(1,2,2)
plt.stem(np.multiply(degree_and_score_mean_corr_statistic,indegree_and_score_mean_corr_pvalue_corrected) )
plt.xticks(np.arange(0,num_networks), labels = visualization_labels, rotation = 70)
plt.ylabel('Significant Correlations with Average NF Score')
plt.grid()
plt.title('Degree Centrality')
plt.ylim([-0.15,0.15])

In [ ]:
data = pd.DataFrame({
    'Label': visualization_labels,
    'Value': degree_and_score_mean_corr_statistic
})

# Plot the data
plt.figure(figsize=(12, 8))
ax = sns.barplot(x='Label', y='Value', data=data, color='orange', width = 0.4)

# Customize labels and title
plt.ylabel('Correlation of Degree and Mean Score', fontsize = 22)
plt.grid(True)
plt.xticks(rotation=60, fontsize = 18)
plt.yticks(fontsize = 18)
plt.xlabel('')
x_left, x_right = plt.xlim()
plt.xlim(-0.5, x_right)

for n in range(0,num_networks):
    bar_value = ax.patches[n].get_height()

    x1 = -0.3 + n
    x2 = 0.3 + n

    if bar_value <0 :

        y = bar_value - 0.005
    else:
        y = bar_value + 0.001

    h = 0
    col = 'k'

    if n == 5:
        plt.text((x1+x2)*.5, y+h, "**", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)
    else:
        plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

plt.tight_layout()
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig08b_degree_vs_score.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig08b_degree_vs_score.png'),bbox_inches='tight')

### Net flow

In [ ]:
causal_flow_and_score_change_corr_statistic = []
causal_flow_and_score_change_corr_pvalue = []

causal_flow_and_score_mean_corr_statistic = []
causal_flow_and_score_mean_corr_pvalue = []

for n in range(0,num_networks):

    causal_flow = np.mean(np.array(all_studies_nf_subnetworks)[:,n,:], axis = 1)- np.mean(np.array(all_studies_nf_subnetworks)[:,:,n], axis = 1)
    
    temp1 = scipy.stats.spearmanr(causal_flow,all_studies_nf_score_changes, alternative='two-sided', nan_policy='omit')
    temp1_statistic = temp1.statistic
    temp1_pvalue = temp1.pvalue
    
    causal_flow_and_score_change_corr_statistic.append(temp1_statistic)
    causal_flow_and_score_change_corr_pvalue.append(temp1_pvalue)

    temp2 = scipy.stats.spearmanr(causal_flow,all_studies_nf_score_mean, alternative='two-sided', nan_policy='omit')
    temp2_statistic = temp2.statistic
    temp2_pvalue = temp2.pvalue

    causal_flow_and_score_mean_corr_statistic.append(temp2_statistic)
    causal_flow_and_score_mean_corr_pvalue.append(temp2_pvalue)

causal_flow_and_score_change_corr_pvalue_corrected = statsmodels.stats.multitest.fdrcorrection(causal_flow_and_score_change_corr_pvalue)[0]
causal_flow_and_score_mean_corr_pvalue_corrected = statsmodels.stats.multitest.fdrcorrection(causal_flow_and_score_mean_corr_pvalue)[0]

plt.figure(figsize = (15,5))
plt.subplot(1,2,1)
plt.stem(np.multiply(causal_flow_and_score_change_corr_statistic,causal_flow_and_score_change_corr_pvalue_corrected) )
plt.xticks(np.arange(0,num_networks), labels = visualization_labels, rotation = 70)
plt.ylabel('Significant Correlations with Changes in NF Score')
plt.grid()
plt.title('Causal Flow')
plt.ylim([-0.15,0.15])

plt.subplot(1,2,2)
plt.stem(np.multiply(causal_flow_and_score_mean_corr_statistic,causal_flow_and_score_mean_corr_pvalue_corrected) )
plt.xticks(np.arange(0,num_networks), labels = visualization_labels, rotation = 70)
plt.ylabel('Significant Correlations with Average NF Score')
plt.grid()
plt.title('Causal Flow')
plt.ylim([-0.15,0.15])

In [ ]:
data = pd.DataFrame({
    'Label': visualization_labels,
    'Value': causal_flow_and_score_mean_corr_statistic
})

# Plot the data
plt.figure(figsize=(12, 8))
ax = sns.barplot(x='Label', y='Value', data=data, color='orange', width = 0.4)

# Customize labels and title
plt.ylabel('Correlation of Net Flow and Mean Score', fontsize = 22)
plt.grid(True)
plt.xticks(rotation=60, fontsize = 18)
plt.yticks(fontsize = 18)
plt.xlabel('')
x_left, x_right = plt.xlim()
plt.xlim(-0.5, x_right)

for n in range(0,num_networks):
    bar_value = ax.patches[n].get_height()
    x1 = -0.3 + n
    x2 = 0.3 + n

    if bar_value <0 :

        y = bar_value - 0.006
    else:
        y = bar_value + 0.001

    h = 0
    col = 'k'

    plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = 14)

plt.tight_layout()
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig08c_net_flow_vs_score.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig08c_net_flow_vs_score.png'),bbox_inches='tight')

## Edges stronger in NF than baseline, and their correlation with score (Fig. 3b,c)

In [ ]:
Stronger_edges_in_NF_than_DC = np.multiply(NF_Dec_compare_edge_strength_statistics, NF_Dec_compare_edge_strength_pvalue_corrected)
Weaker_edges_in_NF_than_DC = np.multiply(NF_Dec_compare_edge_strength_statistics, NF_Dec_compare_edge_strength_pvalue_corrected)

Stronger_edges_in_NF_than_DC [Stronger_edges_in_NF_than_DC<0] = 0
Weaker_edges_in_NF_than_DC [Weaker_edges_in_NF_than_DC>0] = 0

Edges_Significantly_Positively_Correlated_with_Scores = np.multiply(correlations_with_scores, correlations_with_scores_significance)
Edges_Significantly_Negatively_Correlated_with_Scores = np.multiply(correlations_with_scores, correlations_with_scores_significance)

Edges_Significantly_Positively_Correlated_with_Scores [Edges_Significantly_Positively_Correlated_with_Scores <0] = 0
Edges_Significantly_Negatively_Correlated_with_Scores [Edges_Significantly_Negatively_Correlated_with_Scores > 0] = 0

edges_stronger_in_NF_but_do_not_affect_score = np.zeros((num_networks, num_networks))
edges_stronger_in_NF_also_help_with_score = np.zeros((num_networks, num_networks))
edges_stronger_in_NF_but_hurts_score = np.zeros((num_networks, num_networks))
edges_weaker_in_NF_but_help_with_score = np.zeros((num_networks, num_networks))

for i in range(0,num_networks):
    for j in range(num_networks):
        if Stronger_edges_in_NF_than_DC[i,j] != 0 and Edges_Significantly_Positively_Correlated_with_Scores[i,j] ==0 and Edges_Significantly_Negatively_Correlated_with_Scores[i,j]== 0 :
            edges_stronger_in_NF_but_do_not_affect_score[i,j] = 1

        elif Stronger_edges_in_NF_than_DC[i,j] != 0 and Edges_Significantly_Positively_Correlated_with_Scores[i,j] != 0  :
            edges_stronger_in_NF_also_help_with_score [i,j] = 1 

        elif Stronger_edges_in_NF_than_DC[i,j] != 0 and Edges_Significantly_Negatively_Correlated_with_Scores[i,j] != 0  :
            edges_stronger_in_NF_but_hurts_score[i,j] = 1

        elif Weaker_edges_in_NF_than_DC[i,j] != 0 and Edges_Significantly_Positively_Correlated_with_Scores[i,j] != 0 : 
            edges_weaker_in_NF_but_help_with_score[i,j] = 1
        

### Correlation with scores computed based on parcel-level graphs

In [ ]:
parcel_level_correlations_with_scores = np.zeros((num_parcels, num_parcels))
parcel_level_correlations_with_scores_pvalue = np.zeros((num_parcels, num_parcels))

for i in range(0,num_parcels):
    for j in range(0,num_parcels):

        t = scipy.stats.spearmanr(np.array(all_studies_nf_partial_corr)[:,i,j],all_studies_nf_score_mean, alternative='two-sided', nan_policy='omit')
        parcel_level_correlations_with_scores[i,j] = t.statistic
        parcel_level_correlations_with_scores_pvalue[i,j] = t.pvalue
 
parcel_level_correlations_with_scores_significance = np.reshape(statsmodels.stats.multitest.fdrcorrection(parcel_level_correlations_with_scores_pvalue.flatten())[0], (num_parcels, num_parcels))

### NF vs baseline at the parcel level (Fig. 3c, Supplementary Fig. 7)

In [ ]:
parcel_level_NF_Dec_compare_edge_strength_statistics = np.zeros((num_parcels, num_parcels))
parcel_level_NF_Dec_compare_edge_strength_pvalue = np.zeros((num_parcels, num_parcels))

for i in range(0,num_parcels):
    for j in range(0, num_parcels):

        x = np.array(all_studies_nf_partial_corr)[:,i,j]
        y = np.array(all_studies_decoder_partial_correlations)[:,i,j]

        temp = scipy.stats.ranksums(x, y, alternative='two-sided')
        parcel_level_NF_Dec_compare_edge_strength_statistics[i,j] = temp.statistic
        parcel_level_NF_Dec_compare_edge_strength_pvalue[i,j]= temp.pvalue

parcel_level_NF_Dec_compare_edge_strength_pvalue_corrected = np.reshape(statsmodels.stats.multitest.fdrcorrection(parcel_level_NF_Dec_compare_edge_strength_pvalue.flatten())[0], (num_parcels, num_parcels))

In [ ]:
parcel_level_Stronger_edges_in_NF_than_DC = np.multiply(parcel_level_NF_Dec_compare_edge_strength_statistics, parcel_level_NF_Dec_compare_edge_strength_pvalue_corrected)
parcel_level_Weaker_edges_in_NF_than_DC = np.multiply(parcel_level_NF_Dec_compare_edge_strength_statistics, parcel_level_NF_Dec_compare_edge_strength_pvalue_corrected)

parcel_level_Stronger_edges_in_NF_than_DC [parcel_level_Stronger_edges_in_NF_than_DC<0] = 0
parcel_level_Weaker_edges_in_NF_than_DC [parcel_level_Weaker_edges_in_NF_than_DC>0] = 0

parcel_level_Edges_Significantly_Positively_Correlated_with_Scores = np.multiply(parcel_level_correlations_with_scores, parcel_level_correlations_with_scores_significance)
parcel_level_Edges_Significantly_Negatively_Correlated_with_Scores = np.multiply(parcel_level_correlations_with_scores, parcel_level_correlations_with_scores_significance)
parcel_level_Edges_Significantly_Positively_Correlated_with_Scores [parcel_level_Edges_Significantly_Positively_Correlated_with_Scores <0] = 0
parcel_level_Edges_Significantly_Negatively_Correlated_with_Scores [parcel_level_Edges_Significantly_Negatively_Correlated_with_Scores > 0] = 0

parcel_level_edges_stronger_in_NF_but_do_not_affect_score = np.zeros((num_parcels, num_parcels))
parcel_level_edges_stronger_in_NF_also_help_with_score = np.zeros((num_parcels, num_parcels))
parcel_level_edges_stronger_in_NF_but_hurts_score = np.zeros((num_parcels, num_parcels))
parcel_level_edges_weaker_in_NF_but_help_with_score = np.zeros((num_parcels, num_parcels))

for i in range(0,num_parcels):
    for j in range(num_parcels):
        if parcel_level_Stronger_edges_in_NF_than_DC[i,j] != 0 and parcel_level_Edges_Significantly_Positively_Correlated_with_Scores[i,j] ==0 and parcel_level_Edges_Significantly_Negatively_Correlated_with_Scores[i,j]== 0 :
            parcel_level_edges_stronger_in_NF_but_do_not_affect_score[i,j] = 1

        elif parcel_level_Stronger_edges_in_NF_than_DC[i,j] != 0 and parcel_level_Edges_Significantly_Positively_Correlated_with_Scores[i,j] != 0  :
            parcel_level_edges_stronger_in_NF_also_help_with_score [i,j] = 1 

        elif parcel_level_Stronger_edges_in_NF_than_DC[i,j] != 0 and parcel_level_Edges_Significantly_Negatively_Correlated_with_Scores[i,j] != 0  :
            parcel_level_edges_stronger_in_NF_but_hurts_score[i,j] = 1

        elif parcel_level_Weaker_edges_in_NF_than_DC[i,j] != 0 and parcel_level_Edges_Significantly_Positively_Correlated_with_Scores[i,j] != 0 : 
            parcel_level_edges_weaker_in_NF_but_help_with_score[i,j] = 1
        

In [ ]:
plt.figure(figsize = (5,5))
ax1 = plt.subplot()

control_network_parcel_labels = parcel_labels_rearranged[38:38+13]     # control
print(control_network_parcel_labels)
control_network_parcel_labels_shorter = []
for i in range(0,len(control_network_parcel_labels)):
        
        control_network_parcel_labels_shorter.append(control_network_parcel_labels[i][0:2]+control_network_parcel_labels[i][7:])
        

im = plt.imshow(parcel_level_edges_stronger_in_NF_also_help_with_score[38:38+13,38:38+13], cmap = 'Greens')    # control network
ax1.set_xticks(np.arange(len(parcel_labels_rearranged[38:38+13])), labels = control_network_parcel_labels_shorter , fontsize = 12)
ax1.set_yticks(np.arange(len(parcel_labels_rearranged[38:38+13])), labels = control_network_parcel_labels_shorter , fontsize = 12)
g = plt.setp(ax1.get_xticklabels(), rotation =70, ha = "right",
        rotation_mode="anchor")

In [ ]:
plt.figure(figsize = (10,10))
ax1 = plt.subplot()

control_and_dmn_parcels_start = 38
control_and_dmn_parcels_end = control_and_dmn_parcels_start + 4+ 9 + 13+ 11

control_dmn_network_parcel_labels = parcel_labels_rearranged[control_and_dmn_parcels_start:control_and_dmn_parcels_end]     # control
control_dmn_network_parcel_labels_shorter = []
for i in range(0,len(control_dmn_network_parcel_labels)):
        
        if control_dmn_network_parcel_labels[i][3]== 'C':
                control_dmn_network_parcel_labels_shorter.append(control_dmn_network_parcel_labels[i][0:2]+control_dmn_network_parcel_labels[i][7:])
        elif control_dmn_network_parcel_labels[i][3]== 'D':
                control_dmn_network_parcel_labels_shorter.append(control_dmn_network_parcel_labels[i][0:2]+control_dmn_network_parcel_labels[i][10:])
        

im = plt.imshow(parcel_level_edges_stronger_in_NF_also_help_with_score[control_and_dmn_parcels_start:control_and_dmn_parcels_end,control_and_dmn_parcels_start:control_and_dmn_parcels_end], cmap = 'Greens')    # control network
ax1.set_xticks(np.arange(len(parcel_labels_rearranged[control_and_dmn_parcels_start:control_and_dmn_parcels_end])), labels = control_dmn_network_parcel_labels , fontsize = 8)
ax1.set_yticks(np.arange(len(parcel_labels_rearranged[control_and_dmn_parcels_start:control_and_dmn_parcels_end])), labels = control_dmn_network_parcel_labels , fontsize = 8)
g = plt.setp(ax1.get_xticklabels(), rotation =90, ha = "right",
        rotation_mode="anchor")

print('Connections that are stonger in NF and help with score')

plt.savefig(os.path.join(FIG_SUPP, 'SuppFig07_control_hub_with_DMN.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_SUPP, 'SuppFig07_control_hub_with_DMN.png'),bbox_inches='tight')

In [ ]:
# Create a NetworkX graph from the adjacency matrix
G_parcel_level_edges_stronger_in_NF_also_help_with_score = nx.from_numpy_array(parcel_level_edges_stronger_in_NF_also_help_with_score[38:38+13,38:38+13],create_using=nx.DiGraph)
G_parcel_level_edges_stronger_in_NF_but_do_not_affect_score = nx.from_numpy_array(parcel_level_edges_stronger_in_NF_but_do_not_affect_score[38:38+13,38:38+13],create_using=nx.DiGraph)

node_size = 3000
# Plotting parameters
figure = plt.figure(figsize=(12, 12))
ax = plt.subplot()
pos = nx.spring_layout(G_parcel_level_edges_stronger_in_NF_also_help_with_score)

print(control_network_parcel_labels_shorter)

control_network_parcel_labels_shorter_no_L_R = []

for l in range(len(control_network_parcel_labels_shorter)):
    temp = control_network_parcel_labels_shorter[l][3:]

    if l== 0 or l == 1 or l==2 or l ==3 or l == 10 or l== 11 or l== 12:
        control_network_parcel_labels_shorter_no_L_R.append(temp[:-2])
    else:

        control_network_parcel_labels_shorter_no_L_R.append(temp)

print(control_network_parcel_labels_shorter_no_L_R)
pos[0]=[-0.5,-1.5]    #LH_Par_1
pos[1]=[-0.5,2.2]    #LH_PFCl_1
pos[2]=[-.0,-0.3]    #LH_pCun_1
pos[3]=[0.1,0.6]    #LH_Cing_1
pos[4]=[1.5,-1.5]    #RH_Par_1
pos[5]=[1.5,-2]    #RH_Par_2
pos[6]= [1.3,2.2]  #RH_PFCl_1
pos[7]=[1.26,1.15]    #RH_PFCl_2
pos[8]=[1.06,1.2]    #RH_PFCl_3
pos[9]= [1.22,0.75]     #RH_PFCl_4
pos[10]=[0.7,0.6]   #RH_Cing_1
pos[11]=[0.7,1.7]   #RH_PFCmp_1
pos[12]=[0.8,-0.3]   #RH_pCun_1

node_colors = 'lightgray'

nx.draw_networkx_edges(G_parcel_level_edges_stronger_in_NF_also_help_with_score, pos=pos, edge_color='red', width=5, node_size=node_size, arrows=True, arrowsize=12)
nx.draw_networkx_edges(G_parcel_level_edges_stronger_in_NF_but_do_not_affect_score, pos=pos, edge_color='gray', width=2, node_size=node_size, arrows=True, arrowsize=12)

nx.draw_networkx_nodes(G_parcel_level_edges_stronger_in_NF_also_help_with_score, pos=pos, node_color=node_colors, node_size=node_size, linewidths = 3)

node_labels_dict = {i: label for i, label in enumerate(control_network_parcel_labels_shorter_no_L_R)}
nx.draw_networkx_labels(G_parcel_level_edges_stronger_in_NF_also_help_with_score, pos=pos, labels=node_labels_dict, font_size=14, font_color='black', font_weight='bold')

# Set the color, style, and thickness for each spine
# for spine in ax.spines.values():
#     spine.set_edgecolor(box_color)
#     spine.set_linestyle(box_style)
#     spine.set_linewidth(box_thickness)

plt.savefig(os.path.join(FIG_MAIN, 'Fig3c_control_hub_parcels.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig3c_control_hub_parcels.png'),bbox_inches='tight')

In [ ]:
node_locs = np.array([[-15, -40, 10],[15, -40, 10], #sub
                      [-22, -90, 10],[22, -90, 10],  # Vis
                      [-35, -60, 10],[35, -60, 10], # Lim
                      [-10, 50, 10],[10, 50, 10],  # Cont
                      [-30, 30, 10],[30, 30, 10],  # DMN
                      [-40, -0, 10],[40, -0, 10], # Vent
                      [-10, -0, 10],[10, -0, 10], # Dors
                      [-35, -20, 10],[35, -20, 10], # Som-Mot
                      ])

node_positions_2d = node_locs[:, :2]

# Create a NetworkX graph from the adjacency matrix
G_stronger_NF_no_effect_on_score = nx.from_numpy_array(edges_stronger_in_NF_but_do_not_affect_score,create_using=nx.DiGraph)
G_stronger_NF_help_on_score = nx.from_numpy_array(edges_stronger_in_NF_also_help_with_score + edges_stronger_in_NF_but_hurts_score,create_using=nx.DiGraph)
G_stronger_NF_hurt_on_score = nx.from_numpy_array(edges_stronger_in_NF_but_hurts_score,create_using=nx.DiGraph)

print(G_stronger_NF_help_on_score)
node_size = 4000

# Plotting parameters
figure = plt.figure(figsize=(10, 10))
display_mode = 'z'

# Plot glass brain in orthographic view
display = plotting.plot_glass_brain(None, display_mode=display_mode,
                                    figure=figure, annotate=False, alpha=0.15)

node_positions = {i: (x, y) for i, (x, y) in enumerate(node_positions_2d)}

help_edges_with =  5
other_edges_width = 5
nx.draw_networkx_edges(G_stronger_NF_no_effect_on_score, pos=node_positions, edge_color='gray', width=other_edges_width, node_size=node_size, arrows=True, arrowsize=12)
nx.draw_networkx_edges(G_stronger_NF_help_on_score, pos=node_positions, edge_color='red', width=[other_edges_width,other_edges_width,help_edges_with,help_edges_with,other_edges_width,other_edges_width,other_edges_width,other_edges_width,other_edges_width], node_size=node_size, arrows=True, arrowsize=12)
nx.draw_networkx_edges(G_stronger_NF_hurt_on_score, pos=node_positions, edge_color='blue', width=other_edges_width, node_size=node_size, arrows=True, arrowsize=12)

print(G_stronger_NF_help_on_score.edges)
# # Draw nodes
nx.draw_networkx_nodes(G_stronger_NF_hurt_on_score, pos=node_positions, node_color='lightgray', node_size=node_size)

node_labels_dict = {i: label for i, label in enumerate(short_label_modified)}
nx.draw_networkx_labels(G_stronger_NF_hurt_on_score, pos=node_positions, labels=node_labels_dict, font_size=18, font_color='black', font_weight='bold')

plt.text(-15, 67, 'L', fontsize=26, ha='center', va='center', fontweight = 'bold')
plt.text(20, 67, 'R', fontsize=26, ha='center', va='center', fontweight = 'bold')

# Add a rectangle to the plot
rect_x, rect_y = -20, 33  # Bottom-left corner of the rectangle
rect_width, rect_height = 40,30  # Width and height of the rectangle
color = 'brown'  # Color of the rectangle

# Create and add a rectangle patch
rectangle = Rectangle((rect_x, rect_y), rect_width, rect_height, color=color, fill=False, linestyle='--', linewidth=3)
plt.gca().add_patch(rectangle)

plt.savefig(os.path.join(FIG_MAIN, 'Fig3b_modulated_edges_vs_score.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig3b_modulated_edges_vs_score.png'),bbox_inches='tight')

### Asymmetry of baseline, NF and NF minus baseline graphs (Fig. 1e)

In [ ]:
def compute_asymmetry_index(matrix):
    # Ensure the matrix is binary

    # Compute the transpose of the matrix
    transpose_matrix = matrix.T

    # Compute the numerator |A - A^T|
    numerator = np.sum(abs(matrix - transpose_matrix))

    # Compute the denominator |A + A^T|
    denominator = np.sum(abs(matrix + transpose_matrix))

    # Calculate the asymmetry index
    asymmetry_index = numerator / denominator

    return asymmetry_index

In [ ]:
study1_nf_minus_decoder_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
assymmetry_study1_nf_minus_decoder_subnetwork_graphs = []
assymmetry_study1_nf_subnetwork_graphs = []

for sub in range(0,num_subjects):
    average_dec_graph = np.mean(np.array(study1_decoder_subnetwork_graphs)[sub,0,:,:,:], axis = 0)
    for ses in range(0, num_nf_sessions):
        for r in range(0,num_nf_runs):

            nf_graph = study1_nf_subnetwork_graphs[sub][ses][r]
            nf_minus_dec = nf_graph - average_dec_graph
            study1_nf_minus_decoder_subnetwork_graphs[sub][ses][r] = nf_minus_dec
            assymmetry_study1_nf_minus_decoder_subnetwork_graphs.append(compute_asymmetry_index(nf_minus_dec))
            assymmetry_study1_nf_subnetwork_graphs.append(compute_asymmetry_index(nf_graph))

study2_nf_minus_decoder_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
assymmetry_study2_nf_minus_decoder_subnetwork_graphs = []
assymmetry_study2_nf_subnetwork_graphs = []

for sub in range(0,num_subjects):
    average_dec_graph = np.mean(np.array(study2_decoder_subnetwork_graphs)[sub,0,:,:,:], axis = 0)
    for ses in range(0, num_nf_sessions):
        for r in range(0,num_nf_runs):

            nf_graph = study2_nf_subnetwork_graphs[sub][ses][r]
            nf_minus_dec = nf_graph - average_dec_graph
            study2_nf_minus_decoder_subnetwork_graphs[sub][ses][r] = nf_minus_dec
            assymmetry_study2_nf_minus_decoder_subnetwork_graphs.append(compute_asymmetry_index(nf_minus_dec))
            assymmetry_study2_nf_subnetwork_graphs.append(compute_asymmetry_index(nf_graph))

study3_nf_minus_decoder_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
assymmetry_study3_nf_minus_decoder_subnetwork_graphs = []
assymmetry_study3_nf_subnetwork_graphs = []

for sub in range(0,num_subjects):
    average_dec_graph = np.mean(np.array(study3_decoder_subnetwork_graphs)[sub,0,:,:,:], axis = 0)
    for ses in range(0, num_nf_sessions):
        for r in range(0,num_nf_runs):

            nf_graph = study3_nf_subnetwork_graphs[sub][ses][r]
            nf_minus_dec = nf_graph - average_dec_graph
            study3_nf_minus_decoder_subnetwork_graphs[sub][ses][r] = nf_minus_dec
            assymmetry_study3_nf_minus_decoder_subnetwork_graphs.append(compute_asymmetry_index(nf_minus_dec))
            assymmetry_study3_nf_subnetwork_graphs.append(compute_asymmetry_index(nf_graph))

study4_nf_minus_decoder_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
assymmetry_study4_nf_minus_decoder_subnetwork_graphs = []
assymmetry_study4_nf_subnetwork_graphs = []

for sub in range(0,num_subjects):
    average_dec_graph = np.mean(np.array(study4_decoder_subnetwork_graphs)[sub,0,:,:,:], axis = 0)
    for ses in range(0, num_nf_sessions):
        for r in range(0,num_nf_runs):

            nf_graph = study4_nf_subnetwork_graphs[sub][ses][r]
            nf_minus_dec = nf_graph - average_dec_graph
            study4_nf_minus_decoder_subnetwork_graphs[sub][ses][r] = nf_minus_dec
            assymmetry_study4_nf_minus_decoder_subnetwork_graphs.append(compute_asymmetry_index(nf_minus_dec))
            assymmetry_study4_nf_subnetwork_graphs.append(compute_asymmetry_index(nf_graph))

study5_nf_minus_decoder_subnetwork_graphs = [[ [ 0 for i in range(num_nf_runs)] for i in range(num_nf_sessions)] for j in range(num_subjects_ready)] 
assymmetry_study5_nf_minus_decoder_subnetwork_graphs = []
assymmetry_study5_nf_subnetwork_graphs = []

for sub in range(0,num_subjects):
    average_dec_graph = np.mean(np.array(study5_decoder_subnetwork_graphs)[sub,0,:,:,:], axis = 0)
    for ses in range(0, num_nf_sessions):
        for r in range(0,num_nf_runs):

            nf_graph = study5_nf_subnetwork_graphs[sub][ses][r]
            nf_minus_dec = nf_graph - average_dec_graph
            study5_nf_minus_decoder_subnetwork_graphs[sub][ses][r] = nf_minus_dec
            assymmetry_study5_nf_minus_decoder_subnetwork_graphs.append(compute_asymmetry_index(nf_minus_dec))
            assymmetry_study5_nf_subnetwork_graphs.append(compute_asymmetry_index(nf_graph))

In [ ]:
all_decoder_asymmetries = []
for k in range(0,np.array(all_studies_decoder_subnetworks).shape[0]):
    dec_graph = np.array(all_studies_decoder_subnetworks)[k,:,:]
    all_decoder_asymmetries.append(compute_asymmetry_index(dec_graph))

In [ ]:
all_nf_minus_dec_asymmetries = np.concatenate((assymmetry_study1_nf_minus_decoder_subnetwork_graphs,assymmetry_study2_nf_minus_decoder_subnetwork_graphs, assymmetry_study3_nf_minus_decoder_subnetwork_graphs, assymmetry_study4_nf_minus_decoder_subnetwork_graphs, assymmetry_study5_nf_minus_decoder_subnetwork_graphs))
all_nf_asymmetries = np.concatenate((assymmetry_study1_nf_subnetwork_graphs,assymmetry_study2_nf_subnetwork_graphs, assymmetry_study3_nf_subnetwork_graphs, assymmetry_study4_nf_subnetwork_graphs, assymmetry_study5_nf_subnetwork_graphs))

plt.figure(figsize = (8,8))
sns.barplot([all_decoder_asymmetries, all_nf_asymmetries, assymmetry_study1_nf_minus_decoder_subnetwork_graphs], palette=['lightgreen','green','gray'], width = 0.3, errorbar='se')
plt.xticks([0,1,2], labels = ['Baseline','NF','NF-Baseline'], fontsize = x_ticks_font)
plt.yticks(fontsize = y_ticks_font)
plt.ylabel('Graph Asymmetry', fontsize = y_label_font)

x1, x2 = 0, 1.8
y, h, col =0.15, 0.001, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "***", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 1, 1.8
y, h, col =0.133, 0.001, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "***", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

x1, x2 = 0, 1
y, h, col =0.08, 0.0005, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h, "n.s.", ha='center', va='bottom', color=col, fontweight = 'bold', fontsize = significance_font)

sns.stripplot([all_decoder_asymmetries, all_nf_asymmetries, assymmetry_study1_nf_minus_decoder_subnetwork_graphs], palette=['green','darkgreen','dimgray'], size = 2)

plt.yticks([0,0.05,0.1,0.15,0.2])
plt.savefig(os.path.join(FIG_MAIN, 'Fig1e_asymmetry.pdf'),bbox_inches='tight')
plt.savefig(os.path.join(FIG_MAIN, 'Fig1e_asymmetry.png'),bbox_inches='tight')

print(stats.ranksums(np.array(all_nf_minus_dec_asymmetries).flatten(), all_nf_asymmetries, alternative= 'greater'))
print(stats.ranksums(np.array(all_nf_minus_dec_asymmetries).flatten(), all_decoder_asymmetries, alternative= 'greater'))
print(stats.ranksums(np.array(all_decoder_asymmetries).flatten(), all_nf_asymmetries, alternative= 'greater'))
print(len(all_nf_minus_dec_asymmetries), len(all_nf_asymmetries), len(all_decoder_asymmetries))